<a href="https://colab.research.google.com/github/Guilherm3rc/Atend-IA/blob/main/Parcial_Eudora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00.0 - Acessos e permições

In [ ]:
import pandas as pd
import numpy as np
import os
import datetime
from tqdm import tqdm
from dateutil import parser
from datetime import date
from datetime import datetime
from datetime import timedelta
from google.colab import auth
from google.colab import data_table
from google.colab import drive
from google.cloud import bigquery

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# **ATENÇÃO - GESTÃO DO DIRETÓRIO**
pasta_raiz = '/content/drive/Shareddrives/GRE - S&OE/Melhoria Contínua & Indicadores/FR EUD/'

In [ ]:
project = 'sandbox-operacoes-supply'
location = 'US'
client = bigquery.Client(project=project, location=location)

In [ ]:
auth.authenticate_user()

In [ ]:
def input_nr_ciclo():
    while True:
        nr_ciclo = input("Digite um ciclo: ")
        try:
            nr_ciclo = int(nr_ciclo)
            return nr_ciclo
        except ValueError:
            print("Por favor, digite um número inteiro válido.")

In [ ]:

# def obter_primeiro_ultimo_dia_mes(data_str):
#     # Converter a string de entrada para o formato de data
#     data = datetime.strptime(data_str, '%Y-%m')

#     # Primeiro dia do mês
#     var_00_0_primeiro_dia_mes = data.replace(day=1)

#     # Último dia do mês (avançando para o próximo mês e retrocedendo 1 dia)
#     var_00_0_ultimo_dia_mes = var_00_0_primeiro_dia_mes.replace(month=var_00_0_primeiro_dia_mes.month % 12 + 1, day=1) - timedelta(days=1)

#     return var_00_0_primeiro_dia_mes, var_00_0_ultimo_dia_mes

In [ ]:
from datetime import datetime, timedelta
import calendar

def ajustar_periodo_semanal_mensal(data_str):
# 1. Converter entrada 'AAAA-MM' para o dia 1º do mês
    data_base = datetime.strptime(data_str, '%Y-%m')

    # 2. Encontrar a segunda-feira da semana anterior
    # weekday() retorna: 0=Segunda, 1=Terça, ..., 6=Domingo
    dia_da_semana = data_base.weekday()

    # Subtraímos o 'dia_da_semana' para chegar na segunda-feira da semana ATUAL
    # E depois subtraímos 7 dias para chegar na segunda-feira da semana ANTERIOR
    data_inicio = data_base - timedelta(days=dia_da_semana + 7)

    # 3. Encontrar o último dia do mês
    ultimo_dia_num = calendar.monthrange(data_base.year, data_base.month)[1]
    data_final = data_base.replace(day=ultimo_dia_num)

    return data_inicio.date(), data_final.date()

In [ ]:
# Função para formatar a coluna
def formatar_numero(x):
    if pd.isnull(x):
        return "0000000000"
    else:
        return str(int(x)).zfill(10)

In [ ]:
# ciclo base
nr_ciclo = input_nr_ciclo()
print("O ciclo inserido é:", nr_ciclo)

Digite um ciclo: 202610
O ciclo inserido é: 202610


In [ ]:
# mes base
data_str = input("Digite o ano e o mês no formato 'AAAA-MM': ")
var_00_0_primeiro_dia_mes, var_00_0_ultimo_dia_mes = ajustar_periodo_semanal_mensal(data_str)
print("Primeiro dia do mês:", var_00_0_primeiro_dia_mes)
print("Último dia do mês:", var_00_0_ultimo_dia_mes)

Digite o ano e o mês no formato 'AAAA-MM': 2026-10
Primeiro dia do mês: 2026-09-21
Último dia do mês: 2026-10-31


In [ ]:
var_00_0_primeiro_dia_mes_string = var_00_0_primeiro_dia_mes.strftime('%Y-%m-%d')
var_00_0_ultimo_dia_mes_string = var_00_0_ultimo_dia_mes.strftime('%Y-%m-%d')

In [ ]:
# data base
data_atual = datetime.today()# + pd.to_timedelta(1, unit='d')

In [ ]:
nr_ciclo_pasta = str(nr_ciclo) + "/"

In [ ]:
data_atual_string = data_atual.strftime('%Y-%m-%d')

In [ ]:
print("a data de referência é:", data_atual_string)

a data de referência é: 2026-10-14


In [ ]:
sub_pasta = f'000 - Bases GCP/'

In [ ]:
pasta_raiz_bases = f'{pasta_raiz}{sub_pasta}'

In [ ]:
pasta_indicadores = f'006 - Indicadores/'

In [ ]:
caminho_nr_ciclo = os.path.join(pasta_raiz_bases, nr_ciclo_pasta)
caminho_data_atual = os.path.join(caminho_nr_ciclo, data_atual_string)
caminho_indicadores_ciclo = os.path.join(pasta_raiz, pasta_indicadores, nr_ciclo_pasta)

In [ ]:
if not os.path.exists(caminho_nr_ciclo):
    try:
        os.makedirs(caminho_nr_ciclo)
        print(f'Pasta {nr_ciclo_pasta} criada em {pasta_raiz_bases}')
    except OSError as e:
        print(f"Erro ao criar a pasta {caminho_nr_ciclo}: {e}")
        print("Por favor, verifique se você tem permissões de escrita para esta pasta no Google Drive.")
else:
    print(f'Pasta {nr_ciclo_pasta} já existe em {pasta_raiz_bases}')

# Verificar se a pasta da data atual existe dentro de nr_ciclo_pasta e criar se não existe
if not os.path.exists(caminho_data_atual):
    try:
        os.makedirs(caminho_data_atual)
        print(f'Pasta {data_atual_string} criada em {caminho_nr_ciclo}')
    except OSError as e:
        print(f"Erro ao criar a pasta {caminho_data_atual}: {e}")
        print("Por favor, verifique se você tem permissões de escrita para esta pasta no Google Drive.")
else:
    print(f'Pasta {data_atual_string} já existe em {caminho_nr_ciclo}')

Pasta 202610/ já existe em /content/drive/Shareddrives/GRE - S&OE/Melhoria Contínua & Indicadores/FR EUD/000 - Bases GCP/
Pasta 2026-10-14 criada em /content/drive/Shareddrives/GRE - S&OE/Melhoria Contínua & Indicadores/FR EUD/000 - Bases GCP/202610/


In [ ]:
def verificar_e_criar_pasta(caminho):
    if not os.path.exists(caminho):
        os.makedirs(caminho)
        print(f"Pasta {caminho} criada com sucesso!")
    else:
        print(f"Pasta {caminho} já existe.")

In [ ]:
# Verificar e criar a pasta
verificar_e_criar_pasta(caminho_indicadores_ciclo)

Pasta /content/drive/Shareddrives/GRE - S&OE/Melhoria Contínua & Indicadores/FR EUD/006 - Indicadores/202610/ já existe.


In [ ]:
'''
def add_sequence_column(df, base_columns, order_columns, ascending=True):
    """
    Adiciona uma coluna de sequência numerada a um DataFrame, com opção de ordenar em ordem crescente ou decrescente.

    Parâmetros:
    df (DataFrame): O DataFrame de entrada.
    base_columns (list): Lista de colunas que determinam quando a sequência deve ser reiniciada.
    order_columns (list): Lista de colunas usadas para ordenar os dados.
    ascending (bool): Se True, a sequência será ordenada de forma crescente; se False, de forma decrescente.

    Retorna:
    DataFrame: O DataFrame com a nova coluna de sequência adicionada.
    """

    # Ordena o DataFrame com base nas colunas fornecidas e no parâmetro ascending
    df = df.sort_values(by=order_columns, ascending=ascending).reset_index(drop=True)

    # Inicializa a coluna de sequência e as variáveis de controle
    df['sequence'] = 0
    seq = 1
    current_values = {col: None for col in base_columns}

    # Itera sobre as linhas do DataFrame
    for idx, row in df.iterrows():
        # Verifica se as colunas base mudaram e reinicia a sequência se necessário
        if any(row[col] != current_values[col] for col in base_columns):
            current_values = {col: row[col] for col in base_columns}
            seq = 1  # Reinicia a sequência

        # Atribui o valor da sequência e incrementa
        df.at[idx, 'sequence'] = seq
        seq += 1

    return df
'''

'\ndef add_sequence_column(df, base_columns, order_columns, ascending=True):\n    """\n    Adiciona uma coluna de sequência numerada a um DataFrame, com opção de ordenar em ordem crescente ou decrescente.\n\n    Parâmetros:\n    df (DataFrame): O DataFrame de entrada.\n    base_columns (list): Lista de colunas que determinam quando a sequência deve ser reiniciada.\n    order_columns (list): Lista de colunas usadas para ordenar os dados.\n    ascending (bool): Se True, a sequência será ordenada de forma crescente; se False, de forma decrescente.\n\n    Retorna:\n    DataFrame: O DataFrame com a nova coluna de sequência adicionada.\n    """\n\n    # Ordena o DataFrame com base nas colunas fornecidas e no parâmetro ascending\n    df = df.sort_values(by=order_columns, ascending=ascending).reset_index(drop=True)\n\n    # Inicializa a coluna de sequência e as variáveis de controle\n    df[\'sequence\'] = 0\n    seq = 1\n    current_values = {col: None for col in base_columns}\n\n    # Itera 

# 01.0 - Base dos pedidos Full

In [ ]:
# Criando SQL para consulta no GCP
consulta_pedidos = f'''
SELECT
  cod_pedido,
  cod_item_pedido,
  cod_solicitante AS cod_pedido_mar,
  cod_fulfillment,
  cod_item_fulfillment,
  cod_sistema_solicitante,
  tp_condicao_material,
  status_fulfillment,
  status_item_fulfillment,
  status_item_pedido,
  status_item_fulfillment_interno,
  msg_erro_item_fulfillment,
  cod_item_fulfillment_origem,
  cod_item_fulfillment_dest,
  cod_ponto_venda,
  COALESCE(NULLIF(cod_pdv_ecc, ''), cod_pdv_s4) AS cod_pdv_ecc_s4,
  --cod_pdv_ecc,
  --cod_pdv_s4,
  cod_sku_item_fulfillment as cod_sku,
  un_sku,
  qt_original_item_pedido,
  qt_pendente_item_pedido,
  qt_atendida_item_pedido,
  qt_recusada_item_pedido,
  qt_original_item_fulfillment,
  qt_pendente_item_fulfillment,
  qt_solicitada,
  qt_fornecida_item_fulfillment,
  qt_recusada_item_fulfillment,
  dt_atendimento_prevista,
  dt_criacao_pedido,
  dt_movimentacao_pedido,
  dt_atualizacao_pedido,
  ROW_NUMBER() OVER (
    PARTITION BY cod_pedido, cod_item_pedido, cod_item_fulfillment
    ORDER BY dt_movimentacao_pedido
  ) AS rn
FROM `trusted-zone.sellin.tb_pedidos_sellin`
WHERE
  un_sku = 'EUD'
  AND qt_original_item_fulfillment IS NOT NULL
  AND qt_original_item_fulfillment != 0
  AND dt_movimentacao_pedido >= '{var_00_0_primeiro_dia_mes_string}'
  AND dt_atendimento_prevista >= '{var_00_0_primeiro_dia_mes_string}'
  AND dt_atendimento_prevista < cast(current_date as TIMESTAMP)
  AND cod_recusa_item_fulfillment NOT IN ("ZS","08","03","ZQ","06","Z1")
  --AND dt_atendimento_prevista <= '{var_00_0_ultimo_dia_mes_string}'
  --AND cod_pdv_ecc = '0000022780'
  --AND cod_sku = '58664'
  --AND cod_pedido = 1083253
'''

consulta_pedidosp2 = f'''
SELECT
  cod_pedido,
  cod_item_pedido,
  cod_solicitante AS cod_pedido_mar,
  cod_fulfillment,
  cod_item_fulfillment,
  cod_sistema_solicitante,
  tp_condicao_material,
  status_fulfillment,
  status_item_fulfillment,
  status_item_pedido,
  status_item_fulfillment_interno,
  msg_erro_item_fulfillment,
  cod_item_fulfillment_origem,
  cod_item_fulfillment_dest,
  cod_ponto_venda,
  COALESCE(NULLIF(cod_pdv_ecc, ''), cod_pdv_s4) AS cod_pdv_ecc_s4,
  --cod_pdv_ecc,
  --cod_pdv_s4,
  cod_sku_item_fulfillment as cod_sku,
  un_sku,
  qt_original_item_pedido,
  qt_pendente_item_pedido,
  qt_atendida_item_pedido,
  qt_recusada_item_pedido,
  qt_original_item_fulfillment,
  qt_pendente_item_fulfillment,
  qt_solicitada,
  qt_fornecida_item_fulfillment,
  qt_recusada_item_fulfillment,
  dt_atendimento_prevista,
  dt_criacao_pedido,
  dt_movimentacao_pedido,
  dt_atualizacao_pedido,
  ROW_NUMBER() OVER (
    PARTITION BY cod_pedido, cod_item_pedido, cod_item_fulfillment
    ORDER BY dt_movimentacao_pedido
  ) AS rn
FROM `trusted-zone.sellin.tb_pedidos_sellin`
WHERE
  un_sku = 'EUD'
  AND qt_original_item_fulfillment IS NOT NULL
  AND qt_original_item_fulfillment != 0
  AND dt_movimentacao_pedido >= '{var_00_0_primeiro_dia_mes_string}'
  AND dt_atendimento_prevista >= '{var_00_0_primeiro_dia_mes_string}'
  AND dt_atendimento_prevista < cast(current_date as TIMESTAMP)
  --AND dt_atendimento_prevista <= '{var_00_0_ultimo_dia_mes_string}'
  --AND cod_pdv_ecc = '0000022780'
  --AND cod_sku = '58664'
  --AND cod_pedido = 1083253
'''

In [ ]:
# print(consulta_pedidos)

In [ ]:
#df_pedidos_full = pd.read_csv(f'{caminho_data_atual}/df_pedidos.csv')
#df_pedidos_full['cod_pdv_ecc_s4'] = df_pedidos_full['cod_pdv_ecc_s4'].apply(formatar_numero)

In [ ]:
#nulos_por_coluna = df_pedidos_full.isnull().sum()
# Exibindo o resultado
#print(nulos_por_coluna)

In [ ]:
#'''
# Executar a consulta
job = client.query(consulta_pedidos)
df_pedidos_full = job.to_dataframe()

# --- ADICIONE ISTO AQUI ---
# Executar a consulta SEM FILTROS para o cálculo do Atendido
print("Baixando base sem filtros para cálculo de Supply...")
job2 = client.query(consulta_pedidosp2)
df_pedidos_unfiltered = job2.to_dataframe() # Base 'P2' (sem a cláusula WHERE)

# Formatações básicas necessárias para a base unfiltered (mesmas da base full)
df_pedidos_unfiltered['dt_atendimento_prevista'] = pd.to_datetime(df_pedidos_unfiltered['dt_atendimento_prevista']).dt.tz_localize(None)
df_pedidos_unfiltered['dt_movimentacao_pedido'] = pd.to_datetime(df_pedidos_unfiltered['dt_movimentacao_pedido']).dt.tz_localize(None)
df_pedidos_unfiltered['dt_criacao_pedido'] = pd.to_datetime(df_pedidos_unfiltered['dt_criacao_pedido']).dt.date
df_pedidos_unfiltered['qt_fornecida_item_fulfillment'] = df_pedidos_unfiltered['qt_fornecida_item_fulfillment'].fillna(0)
# --------------------------------------------------------------------------

# nome_arquivo = f'df_pedidos.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_pedidos_full.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)
#'''

Baixando base sem filtros para cálculo de Supply...


In [ ]:
# f'{pasta_raiz_saida}{nome_arquivo_saida}'

In [ ]:
df_pedidos_full.dtypes

,0
cod_pedido,Int64
cod_item_pedido,Int64
cod_pedido_mar,object
cod_fulfillment,Int64
cod_item_fulfillment,Int64
cod_sistema_solicitante,object
tp_condicao_material,object
status_fulfillment,object
status_item_fulfillment,object
status_item_pedido,object


In [ ]:
df_pedidos_full[(df_pedidos_full['cod_pdv_ecc_s4'] == '2100000655')].head(30)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,cod_sistema_solicitante,tp_condicao_material,status_fulfillment,status_item_fulfillment,status_item_pedido,...,qt_original_item_fulfillment,qt_pendente_item_fulfillment,qt_solicitada,qt_fornecida_item_fulfillment,qt_recusada_item_fulfillment,dt_atendimento_prevista,dt_criacao_pedido,dt_movimentacao_pedido,dt_atualizacao_pedido,rn
245,3579764,163250349,2513860,4482689,1590076405,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,27,0,27,0,27,2026-10-06 00:00:00+00:00,2026-10-06 00:40:03.496546+00:00,2026-10-13 20:58:48.272304+00:00,2026-10-13 20:58:48.272304+00:00,1
627,3579764,163248667,2513860,4482689,1590075159,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,61,0,61,0,61,2026-10-06 00:00:00+00:00,2026-10-06 00:40:03.496546+00:00,2026-10-13 20:58:48.272304+00:00,2026-10-13 20:58:48.272304+00:00,1
719,3602614,164238886,2527803,4515969,1591562228,MAR-OCBS,REGULAR,PROCESSING,REFUSED,REFUSED,...,43,0,43,0,43,2026-10-13 00:00:00+00:00,2026-10-09 19:51:51.858205+00:00,2026-10-13 20:58:47.252462+00:00,2026-10-13 20:58:47.252462+00:00,1
779,3519805,160505438,2485421,4400126,1586813921,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PARTIAL_BILLED,...,85,0,34,0,34,2026-09-22 00:00:00+00:00,2026-09-19 00:14:17.305613+00:00,2026-10-13 20:58:47.298224+00:00,2026-10-13 20:58:47.298224+00:00,1
1013,3579764,163249843,2513860,4482689,1590075835,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,202,0,202,0,202,2026-10-06 00:00:00+00:00,2026-10-06 00:40:03.496546+00:00,2026-10-13 20:58:48.272304+00:00,2026-10-13 20:58:48.272304+00:00,1
1250,3557740,161901266,2505393,4448993,1588263157,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,27,0,27,0,27,2026-09-29 00:00:00+00:00,2026-09-28 20:55:52.116338+00:00,2026-10-13 20:58:49.278747+00:00,2026-10-13 20:58:49.278747+00:00,1
1251,3557740,161901458,2505393,4448993,1588263436,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,64,0,64,0,64,2026-09-29 00:00:00+00:00,2026-09-28 20:55:52.116338+00:00,2026-10-13 20:58:49.278747+00:00,2026-10-13 20:58:49.278747+00:00,1
1494,3511336,160116797,2479304,4388589,1585819658,MAR-OCBS,REGULAR,REFUSED,REFUSED,REFUSED,...,55,0,55,0,55,2026-09-22 00:00:00+00:00,2026-09-17 21:32:40.323672+00:00,2026-10-12 06:37:14.135220+00:00,2026-09-26 05:28:12.412956+00:00,1
1510,3519805,160506545,2485421,4400126,1586315093,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,76,0,76,0,76,2026-09-22 00:00:00+00:00,2026-09-19 00:14:17.305613+00:00,2026-10-13 20:58:47.298224+00:00,2026-10-13 20:58:47.298224+00:00,1
1617,3557740,161901042,2505393,4448993,1588262832,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,49,0,49,0,49,2026-09-29 00:00:00+00:00,2026-09-28 20:55:52.116338+00:00,2026-10-13 20:58:49.278747+00:00,2026-10-13 20:58:49.278747+00:00,1


In [ ]:
df_pedidos_full[(df_pedidos_full['cod_pdv_ecc_s4'] == '2100000655')].query('cod_sku == "01641"').head(30)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,cod_sistema_solicitante,tp_condicao_material,status_fulfillment,status_item_fulfillment,status_item_pedido,...,qt_original_item_fulfillment,qt_pendente_item_fulfillment,qt_solicitada,qt_fornecida_item_fulfillment,qt_recusada_item_fulfillment,dt_atendimento_prevista,dt_criacao_pedido,dt_movimentacao_pedido,dt_atualizacao_pedido,rn
28572,3602614,164238556,2527803,4515969,1591561895,MAR-OCBS,REGULAR,PROCESSING,REFUSED,REFUSED,...,90,0,90,0,90,2026-10-13 00:00:00+00:00,2026-10-09 19:51:51.858205+00:00,2026-10-13 20:58:47.252462+00:00,2026-10-13 20:58:47.252462+00:00,1
32563,3579764,163248401,2513860,4482689,1590075034,MAR-OCBS,REGULAR,PROCESSING,REFUSED,PENDING,...,73,0,73,0,73,2026-10-06 00:00:00+00:00,2026-10-06 00:40:03.496546+00:00,2026-10-13 20:58:48.272304+00:00,2026-10-13 20:58:48.272304+00:00,1


In [ ]:
df_pedidos_full['dt_atendimento_prevista'] = pd.to_datetime(df_pedidos_full['dt_atendimento_prevista'])
df_pedidos_full['dt_movimentacao_pedido'] = pd.to_datetime(df_pedidos_full['dt_movimentacao_pedido'])
df_pedidos_full['dt_criacao_pedido'] = pd.to_datetime(df_pedidos_full['dt_criacao_pedido'])
df_pedidos_full['dt_atendimento_prevista'] = df_pedidos_full['dt_atendimento_prevista'].dt.tz_localize(None)
df_pedidos_full['dt_movimentacao_pedido'] = df_pedidos_full['dt_movimentacao_pedido'].dt.tz_localize(None)
df_pedidos_full['dt_criacao_pedido'] = df_pedidos_full['dt_criacao_pedido'].dt.date

In [ ]:
df_pedidos_full['qt_original_item_fulfillment']   = df_pedidos_full['qt_original_item_fulfillment'].fillna(0)
df_pedidos_full['qt_pendente_item_fulfillment']   = df_pedidos_full['qt_pendente_item_fulfillment'].fillna(0)
df_pedidos_full['qt_fornecida_item_fulfillment']  = df_pedidos_full['qt_fornecida_item_fulfillment'].fillna(0)
df_pedidos_full['qt_recusada_item_fulfillment']   = df_pedidos_full['qt_recusada_item_fulfillment'].fillna(0)
df_pedidos_full['qt_recusada_item_pedido']        = df_pedidos_full['qt_recusada_item_pedido'].fillna(0)

In [ ]:
df_pedidos_full.shape

(325551, 32)

In [ ]:
# nome_arquivo = f'01_df_pedidos_full.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_pedidos_full.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_pedidos_full[(df_pedidos_full['cod_item_fulfillment'] == 795793658)]# &
           #(df_sugestao_compra_resumo['cod_sku'] == 53439)].head(7)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,cod_sistema_solicitante,tp_condicao_material,status_fulfillment,status_item_fulfillment,status_item_pedido,...,qt_original_item_fulfillment,qt_pendente_item_fulfillment,qt_solicitada,qt_fornecida_item_fulfillment,qt_recusada_item_fulfillment,dt_atendimento_prevista,dt_criacao_pedido,dt_movimentacao_pedido,dt_atualizacao_pedido,rn


In [ ]:
df_pedidos_full[(df_pedidos_full['cod_item_fulfillment'] == 1022180) &
              (df_pedidos_full['cod_sku'] == 53434)].head(3)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,cod_sistema_solicitante,tp_condicao_material,status_fulfillment,status_item_fulfillment,status_item_pedido,...,qt_original_item_fulfillment,qt_pendente_item_fulfillment,qt_solicitada,qt_fornecida_item_fulfillment,qt_recusada_item_fulfillment,dt_atendimento_prevista,dt_criacao_pedido,dt_movimentacao_pedido,dt_atualizacao_pedido,rn


In [ ]:
df_pedidos_full_validacao_ori = df_pedidos_full[['cod_pedido', 'cod_sku', 'dt_criacao_pedido', 'qt_original_item_pedido']].drop_duplicates().copy()

In [ ]:
df_erro = df_pedidos_full[
      (df_pedidos_full['status_item_fulfillment_interno'].isin(['ERROR_VALIDATION']))
].copy()

In [ ]:
df_erro = df_erro[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_item_fulfillment', 'cod_ponto_venda', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'msg_erro_item_fulfillment', 'cod_pdv_ecc_s4',
                         'cod_sku', 'un_sku', 'qt_solicitada', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_item_fulfillment', 'dt_movimentacao_pedido', 'dt_atendimento_prevista', 'dt_criacao_pedido', 'rn']].copy()

In [ ]:
df_erro = df_erro.loc[df_erro.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmin()].copy()

In [ ]:
# nome_arquivo = f'99.9_df_erro.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_erro.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_erro[(df_pedidos_full['cod_pedido'] == 1075266) &
              (df_pedidos_full['cod_sku'] == 51065)].head(10)

/tmp/ipykernel_1592/4249980416.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_erro[(df_pedidos_full['cod_pedido'] == 1075266) &


,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,cod_ponto_venda,status_item_fulfillment,status_item_fulfillment_interno,msg_erro_item_fulfillment,cod_pdv_ecc_s4,cod_sku,un_sku,qt_solicitada,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_item_fulfillment,dt_movimentacao_pedido,dt_atendimento_prevista,dt_criacao_pedido,rn


# 02.0 - Outras bases

In [ ]:
# Criando SQL para consulta no GCP
consulta_pdvs = '''
SELECT
  *
FROM
  `trusted-zone.sellin.tb_cliente_eud`
WHERE dt_processamento = (
  SELECT
    MAX(dt_processamento)
  FROM `trusted-zone.sellin.tb_cliente_eud`
  )
  --AND flg_loja_aberta = 'Sim'
  '''

In [ ]:
# Executar a consulta
job = client.query(consulta_pdvs)
df_pdvs = job.to_dataframe()
df_pdvs_S4 = job.to_dataframe()

In [ ]:
df_pdvs = df_pdvs [['cod_pdv', 'cod_de_para', 'cod_cliente_s4','cod_cd', 'cod_uf', 'cod_canal']]
df_pdvs_S4 = df_pdvs [['cod_pdv', 'cod_de_para', 'cod_cliente_s4','cod_cd', 'cod_uf', 'cod_canal']]

df_pdvs_S4['cod_de_para'] = np.where(df_pdvs_S4['cod_cliente_s4'] != 0, df_pdvs_S4['cod_cliente_s4'], df_pdvs_S4['cod_de_para'])
df_pdvs_S4 = df_pdvs_S4.drop_duplicates()

df_pdvs= pd.concat([df_pdvs, df_pdvs_S4], ignore_index=True)

nome_arquivo = f'df_pdvs.csv'
nome_arquivo_saida = f'{nome_arquivo}'
sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
df_pdvs.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)
# Criar uma regra aqui da tabela

In [ ]:
# Criando SQL para consulta no GCP
consulta_sku = f'''
 SELECT
    cod_material,
    des_material,
    cod_un_material,
    cod_marca_material,
    des_marca_material,
    cod_categoria_material,
    des_categoria_material,
    cod_subcategoria_material,
    des_subcategoria_material,
    cod_gestao_mercadologica_material,
    des_gestao_mercadologica_material,
    cod_status_material,
    des_status_material,
    cod_tipo_material,
    des_tipo_material,
    cod_hierarquia_material,
    cod_condicao_material,
    des_condicao_material,
    des_faixa_etaria_material,
    des_genero_material,
    cod_agrup_sap_material,
    des_agrup_sap_material,
    cod_anterior_sap_material,
    dt_lancamento,
    cod_classificacao_abc_material,
    qt_por_caixa,
  FROM
    `trusted-zone.material.tb_material`
  WHERE
    cod_un_material = 'EUD'
    AND flg_un_original = TRUE
    AND cod_tipo_material IN ('FERT','HAWA')
'''

In [ ]:
# Executar a consulta
job = client.query(consulta_sku)
df_sku = job.to_dataframe()

In [ ]:
# Convertendo uma coluna em valores inteiros usando pd.to_numeric()
df_sku['cod_material'] = pd.to_numeric(df_sku['cod_material'], downcast='integer', errors='coerce')

In [ ]:
# Criando SQL para consulta no GCP
consulta_itens_criticos = f'''
SELECT
    cod_un_agr,
    cod_centro,
    cod_material_concat,
    cod_sem_atualizacao,
    dt_regularizacao,
    st_regularizacao,
FROM `sandbox-operacoes-supply.SeOE_TATICOS.tb_itens_criticos_centro`
WHERE
    cod_un_agr = "GKD"
    and cod_centro in ("DS01", "DB01", "D014")
    and flg_material_bloqueado is false
    and tp_analise in ("CRITICO", "REGULARIZA")
    and des_categoria_onegb_agr not in ("Trade Marketing", "Internacional", "Store Design E Visual Merchandising")
'''

In [ ]:
# Executar a consulta
job = client.query(consulta_itens_criticos)
df_itens_criticos = job.to_dataframe()

In [ ]:
df_02_8_itens_criticos = df_itens_criticos

In [ ]:
df_02_8_itens_criticos['cod_sem_atualizacao'] = df_02_8_itens_criticos['cod_sem_atualizacao'].str.replace('S', '')

In [ ]:
df_02_8_itens_criticos[['semana_iso', 'ano']] = df_02_8_itens_criticos['cod_sem_atualizacao'].str.split('/', expand=True)

In [ ]:
df_02_8_itens_criticos = df_02_8_itens_criticos[['cod_centro', 'cod_material_concat', 'semana_iso', 'ano']]

In [ ]:
# Dividir a coluna 'cod_material_concat' e expandir para criar novas colunas
df_split = df_02_8_itens_criticos['cod_material_concat'].str.split('|', expand=True)

In [ ]:
# Adicionar as novas colunas ao DataFrame original
df_02_8_itens_criticos = pd.concat([df_02_8_itens_criticos, df_split], axis=1)

In [ ]:
# Remover a coluna 'cod_material_concat' original
df_02_8_itens_criticos = df_02_8_itens_criticos.drop(columns=['cod_material_concat'])

In [ ]:
# Derivar novas linhas para cada valor na coluna dividida
df_02_8_itens_criticos = df_02_8_itens_criticos.melt(id_vars=['cod_centro', 'semana_iso', 'ano'], value_name='cod_material_concat')

In [ ]:
# Remover colunas adicionais geradas pelo melt
df_02_8_itens_criticos = df_02_8_itens_criticos.drop(columns=['variable'])

In [ ]:
# Remover valores nulos resultantes da divisão
df_02_8_itens_criticos = df_02_8_itens_criticos.dropna(subset=['cod_material_concat'])

In [ ]:
df_02_8_itens_criticos = df_02_8_itens_criticos.drop_duplicates()

In [ ]:
df_02_8_itens_criticos = df_02_8_itens_criticos.rename(columns={'cod_centro':'cod_cd'}).copy()
df_02_8_itens_criticos = df_02_8_itens_criticos.rename(columns={'semana_iso':'semana_iso_pedido'}).copy()
df_02_8_itens_criticos = df_02_8_itens_criticos.rename(columns={'ano':'ano_pedido'}).copy()
df_02_8_itens_criticos = df_02_8_itens_criticos.rename(columns={'cod_material_concat':'cod_sku'}).copy()

In [ ]:
df_02_8_itens_criticos['Crítico'] = 'Sim'

In [ ]:
df_02_8_itens_criticos['semana_iso_pedido'] = pd.to_numeric(df_02_8_itens_criticos['semana_iso_pedido'], downcast='integer', errors='coerce')
df_02_8_itens_criticos['ano_pedido'] = pd.to_numeric(df_02_8_itens_criticos['ano_pedido'], downcast='integer', errors='coerce')
df_02_8_itens_criticos['cod_sku'] = pd.to_numeric(df_02_8_itens_criticos['cod_sku'], downcast='integer', errors='coerce')

In [ ]:
# Criando SQL para consulta no GCP
consulta_preco_calamo = f'''
select
  cod_material,
  vlr_preco_medio,
  FROM `refined-zone.supply.tb_supply_preco_calamo`
  --AND cod_material = '55802'
  --WHERE cod_centro = 'DB01'
  WHERE cod_un_material = 'EUD'
  --AND cod_deposito LIKE '%TE%'
  --AND cod_tipo_material in ('HAWA', 'FERT')
  ORDER BY cod_material
'''

In [ ]:
# Executar a consulta
job = client.query(consulta_preco_calamo)
df_preco_calamo = job.to_dataframe()

In [ ]:
df_preco_calamo['cod_material'] = pd.to_numeric(df_preco_calamo['cod_material'], downcast='integer', errors='coerce')
df_preco_calamo['vlr_preco_medio'] = df_preco_calamo['vlr_preco_medio'].replace('nan', np.nan).astype(float)
df_preco_calamo = df_preco_calamo.rename(columns={'cod_material':'cod_sku'}).copy()

### 02.0.1 Sug Compra

In [ ]:
# Criando SQL para consulta OTM
consulta_sug_compra = f'''
WITH cte_max_date AS (
  SELECT
    cod_material,
    cod_franquia,
    DATE(dt_processamento) AS date_processamento,
    MAX(dt_processamento) AS max_dt_processamento
  FROM `refined-zone.abastecimento.tb_mar_compra_inteligente_hist`
  WHERE dt_processamento >= TIMESTAMP('{var_00_0_primeiro_dia_mes_string}')
  AND dt_processamento <= TIMESTAMP('{var_00_0_ultimo_dia_mes_string}') -- Usar TIMESTAMP para garantir que está usando o particionamento corretamente
  AND cod_un_negocio = 'EUD'
  GROUP BY cod_material, cod_franquia, DATE(dt_processamento)
)

SELECT
  LPAD(CAST(a.cod_material AS STRING), 5, '0') AS cod_material,
  LPAD(CAST(a.cod_franquia AS STRING), 10, '0') AS cod_franquia,
  --SAFE_CAST(a.qt_estoque_disponivel AS INT64) AS qt_estoque_disponivel,
  --SAFE_CAST(a.qt_estoque_transito AS INT64) AS qt_estoque_transito,
  SAFE_CAST(a.qt_sugerida_compra AS INT64) AS qt_sugerida_compra,
  DATE(a.dt_processamento) AS dt_processamento,
  a.qt_dias_cobertura_alvo,
  a.qt_dias_ddv
FROM `refined-zone.abastecimento.tb_mar_compra_inteligente_hist` AS a
JOIN cte_max_date AS b
ON a.cod_material = b.cod_material
AND a.cod_franquia = b.cod_franquia
AND DATE(a.dt_processamento) = b.date_processamento
AND a.dt_processamento = b.max_dt_processamento
WHERE a.dt_processamento >= TIMESTAMP('{var_00_0_primeiro_dia_mes_string}')
AND a.dt_processamento <= TIMESTAMP('{var_00_0_ultimo_dia_mes_string}')  -- Filtro adicional para particionamento
AND cod_un_negocio = 'EUD'
'''

In [ ]:
#df_sug_compra = pd.read_csv(f'{caminho_data_atual}/df_sug_compra.csv')

In [ ]:
#'''
# Executar a consulta
job = client.query(consulta_sug_compra)
df_sug_compra = job.to_dataframe()

# nome_arquivo = f'df_sug_compra.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_sug_compra.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)
#'''

In [ ]:
df_sugestao_compra = df_sug_compra

In [ ]:
df_sugestao_compra

,cod_material,cod_franquia,qt_sugerida_compra,dt_processamento,qt_dias_cobertura_alvo,qt_dias_ddv
0,84972,0000013706,0,2026-10-08,32.0,0.000000
1,84961,0000013706,0,2026-10-08,32.0,0.000000
2,86285,0000012962,0,2026-10-08,31.0,0.053315
3,94051,0000013995,0,2026-10-08,61.0,0.536452
4,51750,0000013537,9,2026-10-08,36.0,0.242601
...,...,...,...,...,...,...
21062652,53500,0000024850,0,2026-10-11,57.0,0.742212
21062653,53501,0000022449,58,2026-10-01,57.0,2.042994
21062654,94051,0000020422,0,2026-10-04,57.0,0.145978
21062655,88303,0000020190,0,2026-10-08,57.0,0.071837


In [ ]:
df_sugestao_compra['dt_processamento'] = pd.to_datetime(df_sugestao_compra['dt_processamento'])

In [ ]:
df_sugestao_compra['dt_atendimento_prevista'] = pd.to_datetime(df_sugestao_compra['dt_processamento'])
# Extrair o ano
df_sugestao_compra['ano_pedido'] = df_sugestao_compra['dt_processamento'].dt.year
# Extrair o mês
df_sugestao_compra['mes_pedido'] = df_sugestao_compra['dt_processamento'].dt.month
# Extrair a semana ISO
df_sugestao_compra['semana_iso_pedido'] = df_sugestao_compra['dt_processamento'].dt.isocalendar().week

In [ ]:
df_sugestao_compra_resumo = df_sugestao_compra.groupby(
    ['cod_franquia', 'cod_material','ano_pedido', 'mes_pedido', 'semana_iso_pedido']
).agg(
    sc_media=('qt_sugerida_compra', 'mean'),
    sc_minimo=('qt_sugerida_compra', 'min'),
    sc_maximo=('qt_sugerida_compra', 'max'),
    qt_dias_cobertura_alvo_media=('qt_dias_cobertura_alvo', 'mean'),
    qt_dias_cobertura_alvo_minimo=('qt_dias_cobertura_alvo', 'min'),
    qt_dias_cobertura_alvo_maximo=('qt_dias_cobertura_alvo', 'max'),
    qt_dias_ddv_media=('qt_dias_ddv', 'mean')
).reset_index().copy()

In [ ]:
df_sugestao_compra_resumo['sc_media'] = df_sugestao_compra_resumo['sc_media'].astype(int)
df_sugestao_compra_resumo['sc_minimo'] = df_sugestao_compra_resumo['sc_minimo'].astype(int)
df_sugestao_compra_resumo['sc_maximo'] = df_sugestao_compra_resumo['sc_maximo'].astype(int)
df_sugestao_compra_resumo['qt_dias_cobertura_alvo_media'] = df_sugestao_compra_resumo['qt_dias_cobertura_alvo_media'].astype(int)
df_sugestao_compra_resumo['qt_dias_cobertura_alvo_minimo'] = df_sugestao_compra_resumo['qt_dias_cobertura_alvo_minimo'].astype(int)
df_sugestao_compra_resumo['qt_dias_cobertura_alvo_maximo'] = df_sugestao_compra_resumo['qt_dias_cobertura_alvo_maximo'].astype(int)
df_sugestao_compra_resumo['qt_dias_ddv_media'] = df_sugestao_compra_resumo['qt_dias_ddv_media'].astype(int)
df_sugestao_compra_resumo['cod_material'] = df_sugestao_compra_resumo['cod_material'].astype(int)

In [ ]:
df_sugestao_compra_resumo

,cod_franquia,cod_material,ano_pedido,mes_pedido,semana_iso_pedido,sc_media,sc_minimo,sc_maximo,qt_dias_cobertura_alvo_media,qt_dias_cobertura_alvo_minimo,qt_dias_cobertura_alvo_maximo,qt_dias_ddv_media
0,0000003396,1062,2026,9,39,0,0,0,40,40,41,0
1,0000003396,1062,2026,9,40,0,0,0,41,41,41,0
2,0000003396,1062,2026,10,40,0,0,0,41,41,41,0
3,0000003396,1062,2026,10,41,1,0,8,41,41,41,0
4,0000003396,1062,2026,10,42,0,0,0,41,41,41,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4635734,0000027519,96397,2026,10,42,0,0,0,29,29,29,0
4635735,0000027519,96617,2026,10,41,10,0,21,29,29,29,0
4635736,0000027519,96617,2026,10,42,2,0,5,29,29,29,0
4635737,0000027519,96626,2026,10,41,1,0,3,29,29,29,0


In [ ]:
df_sugestao_compra_resumo = df_sugestao_compra_resumo.rename(columns={'cod_franquia':'cod_pdv_ecc_s4'}).copy()
df_sugestao_compra_resumo = df_sugestao_compra_resumo.rename(columns={'cod_material':'cod_sku'}).copy()
df_sugestao_compra_resumo = df_sugestao_compra_resumo.rename(columns={'sc_media':'qt_sc_media'}).copy()
df_sugestao_compra_resumo = df_sugestao_compra_resumo.rename(columns={'sc_minimo':'qt_sc_minimo'}).copy()
df_sugestao_compra_resumo = df_sugestao_compra_resumo.rename(columns={'sc_maximo':'qt_sc_maximo'}).copy()

In [ ]:
df_sugestao_compra_resumo

,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,qt_sc_media,qt_sc_minimo,qt_sc_maximo,qt_dias_cobertura_alvo_media,qt_dias_cobertura_alvo_minimo,qt_dias_cobertura_alvo_maximo,qt_dias_ddv_media
0,0000003396,1062,2026,9,39,0,0,0,40,40,41,0
1,0000003396,1062,2026,9,40,0,0,0,41,41,41,0
2,0000003396,1062,2026,10,40,0,0,0,41,41,41,0
3,0000003396,1062,2026,10,41,1,0,8,41,41,41,0
4,0000003396,1062,2026,10,42,0,0,0,41,41,41,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4635734,0000027519,96397,2026,10,42,0,0,0,29,29,29,0
4635735,0000027519,96617,2026,10,41,10,0,21,29,29,29,0
4635736,0000027519,96617,2026,10,42,2,0,5,29,29,29,0
4635737,0000027519,96626,2026,10,41,1,0,3,29,29,29,0


In [ ]:
df_sugestao_compra_resumo['cod_pdv_ecc_s4'] = df_sugestao_compra_resumo['cod_pdv_ecc_s4'].astype(int)
df_sugestao_compra_resumo['cod_sku'] = df_sugestao_compra_resumo['cod_sku'].astype(int)
df_sugestao_compra_resumo['ano_pedido'] = df_sugestao_compra_resumo['ano_pedido'].astype(int)
df_sugestao_compra_resumo['mes_pedido'] = df_sugestao_compra_resumo['mes_pedido'].astype(int)
df_sugestao_compra_resumo['semana_iso_pedido'] = df_sugestao_compra_resumo['semana_iso_pedido'].astype(int)

In [ ]:
#df_sugestao_compra_resumo[(df_sugestao_compra_resumo['cod_item_fulfillment'] == 795793658)]# &
           #(df_sugestao_compra_resumo['cod_sku'] == 53439)].head(7)

# 03.0 - Pedidos criados

In [ ]:
df_pedidos = df_pedidos_full[
    (~df_pedidos_full['status_item_fulfillment'].isin(['CANCELED_BY_FRANCHISEE', 'REPLACED'])) &
    #(df_pedidos_full['rn'] == 1) &
    (df_pedidos_full['qt_original_item_fulfillment'] > 0)
].copy()

In [ ]:
df_pedidos = df_pedidos.loc[df_pedidos.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmax()].copy()

In [ ]:
#df_pedidos[(df_pedidos['cod_item_fulfillment'] == cod_item_fulfillment) &
#           (df_pedidos['cod_sku'] == 53439)].head(7)

In [ ]:
df_pedidos = df_pedidos[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                         'cod_sku', 'un_sku', 'qt_original_item_fulfillment', 'qt_solicitada', 'dt_movimentacao_pedido', 'dt_atendimento_prevista', 'dt_criacao_pedido', 'rn']].copy()

In [ ]:
df_pedidos['dt_atendimento_prevista'] = pd.to_datetime(df_pedidos['dt_atendimento_prevista'])
# Extrair o ano
df_pedidos['ano_pedido'] = df_pedidos['dt_atendimento_prevista'].dt.year
# Extrair o mês
df_pedidos['mes_pedido'] = df_pedidos['dt_atendimento_prevista'].dt.month
# Extrair a semana ISO
df_pedidos['semana_iso_pedido'] = df_pedidos['dt_atendimento_prevista'].dt.isocalendar().week

In [ ]:
df_pedidos['dt_movimentacao_pedido'] = pd.to_datetime(df_pedidos['dt_movimentacao_pedido'])
# Extrair o ano
df_pedidos['ano_movimentacao'] = df_pedidos['dt_movimentacao_pedido'].dt.year
# Extrair o mês
df_pedidos['mes_movimentacao'] = df_pedidos['dt_movimentacao_pedido'].dt.month
# Extrair a semana ISO
df_pedidos['semana_iso_movimentacao'] = df_pedidos['dt_movimentacao_pedido'].dt.isocalendar().week

In [ ]:
# nome_arquivo = f'02_df_pedidos.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_pedidos.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_pedidos[(df_pedidos['cod_pedido'] == 1083253) &
              (df_pedidos['cod_sku'] == 57550)].head(10)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_original_item_fulfillment,...,dt_movimentacao_pedido,dt_atendimento_prevista,dt_criacao_pedido,rn,ano_pedido,mes_pedido,semana_iso_pedido,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao


In [ ]:
df_pedidos[(df_pedidos['cod_pedido'] == 1083253) &
              (df_pedidos['cod_sku'] == 58664)].head(10)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_original_item_fulfillment,...,dt_movimentacao_pedido,dt_atendimento_prevista,dt_criacao_pedido,rn,ano_pedido,mes_pedido,semana_iso_pedido,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao


## 03.1 - Data de atendimento prevista

In [ ]:
# Selecionando as colunas
df_atendimento_prevista = df_pedidos[['cod_pedido', 'cod_item_fulfillment', 'cod_sku', 'dt_atendimento_prevista', 'dt_criacao_pedido']].copy()

In [ ]:
df_atendimento_prevista = df_atendimento_prevista.groupby(['cod_pedido', 'cod_item_fulfillment', 'cod_sku', 'dt_criacao_pedido'])['dt_atendimento_prevista'].min().reset_index()

In [ ]:
# Remover linhas com valores nulos
df_atendimento_prevista = df_atendimento_prevista.dropna()

In [ ]:
# Remover duplicados com base nas colunas cod_pedido e dt_atendimento_prevista
df_atendimento_prevista = df_atendimento_prevista.drop_duplicates(subset=['cod_pedido', 'cod_item_fulfillment', 'cod_sku', 'dt_atendimento_prevista', 'dt_criacao_pedido'])

In [ ]:
# Ordenar o DataFrame com base na coluna dt_atendimento_prevista
df_atendimento_prevista = df_atendimento_prevista.sort_values(by='cod_pedido')

In [ ]:
# Resetar o índice
df_atendimento_prevista = df_atendimento_prevista.reset_index(drop=True)

In [ ]:
df_atendimento_prevista['dt_atendimento_prevista'] = pd.to_datetime(df_atendimento_prevista['dt_atendimento_prevista'])
# Extrair o ano
df_atendimento_prevista['ano_pedido'] = df_atendimento_prevista['dt_atendimento_prevista'].dt.year
# Extrair o mês
df_atendimento_prevista['mes_pedido'] = df_atendimento_prevista['dt_atendimento_prevista'].dt.month
# Extrair a semana ISO
df_atendimento_prevista['semana_iso_pedido'] = df_atendimento_prevista['dt_atendimento_prevista'].dt.isocalendar().week

In [ ]:
# nome_arquivo = f'00_df_atendimento_prevista.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_atendimento_prevista.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_atendimento_prevista

,cod_pedido,cod_item_fulfillment,cod_sku,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido
0,3197590,1575114208,94666,2026-07-09,2026-10-05,2026,10,41
1,3197590,1575256500,94665,2026-07-09,2026-10-05,2026,10,41
2,3197590,1575253460,94670,2026-07-09,2026-10-05,2026,10,41
3,3197590,1575114229,94661,2026-07-09,2026-10-05,2026,10,41
4,3197590,1575114226,94667,2026-07-09,2026-10-05,2026,10,41
...,...,...,...,...,...,...,...,...
317676,3609350,1592002133,53428,2026-10-13,2026-10-13,2026,10,42
317677,3609350,1592002159,59565,2026-10-13,2026-10-13,2026,10,42
317678,3609350,1592002111,47364,2026-10-13,2026-10-13,2026,10,42
317679,3609350,1592002128,52212,2026-10-13,2026-10-13,2026,10,42


# 04.0 - Redutor de pedidos

## 04.1 - Cancelamento do franquiado

In [ ]:
df_cancelado_franchisee = df_pedidos_full[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_item_fulfillment_origem', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                                            'cod_sku', 'un_sku', 'qt_recusada_item_pedido', 'dt_movimentacao_pedido', 'rn']].copy()

In [ ]:
df_cancelado_franchisee[(df_cancelado_franchisee['cod_sku'] == '55559') &
              (df_cancelado_franchisee['cod_pedido'] == 1082494)].head(50)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment_origem,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_recusada_item_pedido,dt_movimentacao_pedido,rn


In [ ]:
df_cancelado_franchisee = df_cancelado_franchisee[
    (df_cancelado_franchisee['status_item_fulfillment'].isin(['CANCELED_BY_FRANCHISEE'])) &
    #(df_cancelado_franchisee['rn'] == 1) &
    (df_cancelado_franchisee['qt_recusada_item_pedido'] > 0)
].copy()

In [ ]:
df_cancelado_franchisee[(df_cancelado_franchisee['cod_sku'] == '55559') &
              (df_cancelado_franchisee['cod_pedido'] == 1082494)].head(50)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment_origem,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_recusada_item_pedido,dt_movimentacao_pedido,rn


In [ ]:
df_cancelado_franchisee = df_cancelado_franchisee.rename(columns={'cod_item_fulfillment_origem':'cod_item_fulfillment'})

In [ ]:
valores_ordenados = df_cancelado_franchisee['cod_sku'].sort_values()
print(valores_ordenados)

112780    01062
112400    01062
265364    01062
60469     01062
213899    01062
          ...  
247542    97815
248719    97815
275774    97815
323135    97815
252049    97815
Name: cod_sku, Length: 7870, dtype: object


In [ ]:
# Usando a função para gerar a sequência
#df_cancelado_franchisee = add_sequence_column(df_cancelado_franchisee, base_columns=['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'], order_columns=['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment', 'dt_movimentacao_pedido'], ascending=False)

In [ ]:
df_cancelado_franchisee = df_cancelado_franchisee[
    (df_cancelado_franchisee['rn'] == 1) &
    (df_cancelado_franchisee['qt_recusada_item_pedido'] > 0)
].copy()

In [ ]:
df_cancelado_franchisee['dt_movimentacao_pedido'] = pd.to_datetime(df_cancelado_franchisee['dt_movimentacao_pedido'])
# Extrair o ano
df_cancelado_franchisee['ano_movimentacao'] = df_cancelado_franchisee['dt_movimentacao_pedido'].dt.year
# Extrair o mês
df_cancelado_franchisee['mes_movimentacao'] = df_cancelado_franchisee['dt_movimentacao_pedido'].dt.month
# Extrair a semana ISO
df_cancelado_franchisee['semana_iso_movimentacao'] = df_cancelado_franchisee['dt_movimentacao_pedido'].dt.isocalendar().week

In [ ]:
df_cancelado_franchisee = df_cancelado_franchisee.merge(df_atendimento_prevista, how='left', left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                              right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
df_cancelado_franchisee.shape

(7870, 20)

In [ ]:
df_cancelado_franchisee = df_cancelado_franchisee.rename(columns={'qt_recusada_item_pedido':'qt_cancelado_franchisee'})

In [ ]:
 nome_arquivo = f'03_1_df_cancelado_franchisee.csv'
 nome_arquivo_saida = f'{nome_arquivo}'
 sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
 pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
 df_cancelado_franchisee.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_cancelado_franchisee[(df_cancelado_franchisee['cod_pedido'] == 1083253) &
                        (df_cancelado_franchisee['cod_sku'] == 57550)].head(10)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_cancelado_franchisee,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


In [ ]:
df_cancelado_franchisee[(df_cancelado_franchisee['cod_pedido'] == 1083253) &
                        (df_cancelado_franchisee['cod_sku'] == 58664)].head(10)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_cancelado_franchisee,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


## 04.2 - Faseamento

In [ ]:
df_phased = df_pedidos_full[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_item_fulfillment', 'cod_item_fulfillment_origem', 'cod_item_fulfillment_dest', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                                            'cod_sku', 'un_sku', 'qt_pendente_item_fulfillment', 'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn']].copy()

In [ ]:
df_phased = df_phased.loc[df_phased.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmax()].copy()

In [ ]:
df_phased[(df_phased['cod_pedido'] == 1018268) &
           (df_phased['cod_sku'] == 52171)].head(7)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,cod_item_fulfillment_origem,cod_item_fulfillment_dest,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_pendente_item_fulfillment,dt_movimentacao_pedido,dt_criacao_pedido,rn


In [ ]:
df_phased = df_phased[
    (df_phased['status_item_fulfillment'].isin(['PHASED'])) |
    (~df_phased['status_item_fulfillment'].isin(['REPLACED'])) &
    #(df_phased['sequence'] == 1) &
    (df_phased['qt_pendente_item_fulfillment'] > 0) &
    (~df_phased['cod_item_fulfillment_dest'].isna() | (df_phased['cod_item_fulfillment_dest'] == ''))
    #(df_phased['cod_item_fulfillment_origem'].isna() | (df_phased['cod_item_fulfillment_origem'] == ''))
].copy()

In [ ]:
df_phased[(df_phased['cod_pedido'] == 1102192) &
           (df_phased['cod_sku'] == 53144)].head(7)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,cod_item_fulfillment_origem,cod_item_fulfillment_dest,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_pendente_item_fulfillment,dt_movimentacao_pedido,dt_criacao_pedido,rn


In [ ]:
df_phased = df_phased[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                                            'cod_sku', 'un_sku', 'qt_pendente_item_fulfillment', 'dt_movimentacao_pedido', 'rn']].copy()

In [ ]:
#df_cancelado_franchisee = df_cancelado_franchisee.rename(columns={'cod_item_fulfillment_origem':'cod_item_fulfillment'})

In [ ]:
df_phased['dt_movimentacao_pedido'] = pd.to_datetime(df_phased['dt_movimentacao_pedido'])
# Extrair o ano
df_phased['ano_movimentacao'] = df_phased['dt_movimentacao_pedido'].dt.year
# Extrair o mês
df_phased['mes_movimentacao'] = df_phased['dt_movimentacao_pedido'].dt.month
# Extrair a semana ISO
df_phased['semana_iso_movimentacao'] = df_phased['dt_movimentacao_pedido'].dt.isocalendar().week

In [ ]:
df_phased = df_phased.merge(df_atendimento_prevista, how='left', left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                              right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
df_phased.shape

(7, 20)

In [ ]:
df_phased = df_phased.rename(columns={'qt_pendente_item_fulfillment':'qt_phased'})

In [ ]:
# nome_arquivo = f'03_2_df_phased.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_phased.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_phased[(df_phased['cod_pedido'] == 1083253) &
          (df_phased['cod_sku'] == 57550)].head(10)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_phased,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


In [ ]:
df_phased[(df_phased['cod_pedido'] == 1083253) &
          (df_phased['cod_sku'] == 58664)].head(10)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_phased,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


# 05.0 - Fornecido

In [ ]:
# 05.0 - Fornecido (CORREÇÃO DE DATAS)

# 1. Criação de um dicionário de datas COMPLETO usando a base Unfiltered
# Isso garante que itens recusados ('ZS', etc) tenham sua data prevista original recuperada
df_datas_full = df_pedidos_unfiltered[['cod_pedido', 'cod_item_fulfillment', 'cod_sku', 'dt_atendimento_prevista', 'dt_criacao_pedido']].copy()

# Remove nulos e duplicatas para ter uma chave única (mesma lógica da seção 03.1)
df_datas_full = df_datas_full.dropna(subset=['dt_atendimento_prevista'])
df_datas_full = df_datas_full.groupby(['cod_pedido', 'cod_item_fulfillment', 'cod_sku', 'dt_criacao_pedido'])['dt_atendimento_prevista'].min().reset_index()

# Extração das dimensões de tempo para a base de datas
df_datas_full['ano_previsto_ref'] = df_datas_full['dt_atendimento_prevista'].dt.year
df_datas_full['mes_previsto_ref'] = df_datas_full['dt_atendimento_prevista'].dt.month
df_datas_full['sem_prevista_ref'] = df_datas_full['dt_atendimento_prevista'].dt.isocalendar().week

# 2. Preparação do DataFrame de Fornecido (Supply)
df_fornecido = df_pedidos_unfiltered[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                             'cod_sku', 'un_sku', 'qt_fornecida_item_fulfillment', 'dt_movimentacao_pedido', 'rn']].copy()

df_fornecido = df_fornecido[
    (df_fornecido['qt_fornecida_item_fulfillment'] > 0)
].copy()

# Pega o último registro válido
df_fornecido = df_fornecido.loc[df_fornecido.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmax()].copy()

# Tratamento de datas de movimentação
df_fornecido['dt_movimentacao_pedido'] = pd.to_datetime(df_fornecido['dt_movimentacao_pedido'])
df_fornecido['ano_movimentacao'] = df_fornecido['dt_movimentacao_pedido'].dt.year
df_fornecido['mes_movimentacao'] = df_fornecido['dt_movimentacao_pedido'].dt.month
df_fornecido['semana_iso_movimentacao'] = df_fornecido['dt_movimentacao_pedido'].dt.isocalendar().week

# 3. O MERGE CRUCIAL
# Agora cruzamos com df_datas_full (criado acima) e não com df_atendimento_prevista (que vem da base filtrada)
df_fornecido = df_fornecido.merge(df_datas_full, how='left',
                                  left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                                  right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

# 4. Atribuição das Colunas Finais
# Agora usamos as colunas de referência do df_datas_full
df_fornecido['ano_pedido'] = df_fornecido['ano_previsto_ref']
df_fornecido['mes_pedido'] = df_fornecido['mes_previsto_ref']
df_fornecido['semana_iso_pedido'] = df_fornecido['sem_prevista_ref']

# Caso ainda sobre algum nulo (muito raro agora), usamos a movimentação como fallback
df_fornecido['dt_atendimento_prevista'] = df_fornecido['dt_atendimento_prevista'].fillna(df_fornecido['dt_movimentacao_pedido'])
df_fornecido['ano_pedido'] = df_fornecido['ano_pedido'].fillna(df_fornecido['ano_movimentacao'])
df_fornecido['mes_pedido'] = df_fornecido['mes_pedido'].fillna(df_fornecido['mes_movimentacao'])
df_fornecido['semana_iso_pedido'] = df_fornecido['semana_iso_pedido'].fillna(df_fornecido['semana_iso_movimentacao'])

# Limpeza de colunas auxiliares
df_fornecido.drop(columns=['ano_previsto_ref', 'mes_previsto_ref', 'sem_prevista_ref'], inplace=True, errors='ignore')

print(f"Shape final df_fornecido: {df_fornecido.shape}")

Shape final df_fornecido: (452847, 20)


# 06.0 - Pendente

In [ ]:
df_pendente = df_pedidos_full[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_fulfillment', 'cod_item_fulfillment', 'status_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                             'cod_sku', 'un_sku', 'qt_solicitada','qt_pendente_item_fulfillment', 'dt_movimentacao_pedido', 'rn']].copy()

In [ ]:
df_pendente = df_pendente.loc[df_pendente.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmax()].copy()

In [ ]:
# Usando a função para gerar a sequência
#df_pendente = add_sequence_column(df_pendente, base_columns=['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'], order_columns=['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment', 'dt_movimentacao_pedido'], ascending=False)

In [ ]:
df_pendente = df_pendente[
      (~df_pendente['status_item_fulfillment'].isin(['PHASED']))
].copy()

In [ ]:
df_pendente = df_pendente[
      (df_pendente['status_item_fulfillment'].isin(['PROCESSING', 'OPEN']))
].copy()

In [ ]:
df_pendente['dt_movimentacao_pedido'] = pd.to_datetime(df_pendente['dt_movimentacao_pedido'])
# Extrair o ano
df_pendente['ano_movimentacao'] = df_pendente['dt_movimentacao_pedido'].dt.year
# Extrair o mês
df_pendente['mes_movimentacao'] = df_pendente['dt_movimentacao_pedido'].dt.month
# Extrair a semana ISO
df_pendente['semana_iso_movimentacao'] = df_pendente['dt_movimentacao_pedido'].dt.isocalendar().week

In [ ]:
df_pendente = df_pendente.merge(df_atendimento_prevista, how='left', left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                              right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
df_pendente[(df_pendente['cod_sku'] == 58406) & #57550
              (df_pendente['cod_pedido'] == 1083253)].head(50)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,status_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,...,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


In [ ]:
# nome_arquivo = f'05_df_pendente.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_pendente.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

# 07.0 - Recusado 30 dias

In [ ]:
df_recusada30 = df_pedidos_full[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_fulfillment', 'cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                                 'cod_sku', 'un_sku', 'qt_recusada_item_fulfillment', 'qt_recusada_item_pedido', 'dt_movimentacao_pedido', 'rn']].copy()

In [ ]:
df_recusada30[(df_recusada30['cod_sku'] == '55559') &
              (df_recusada30['cod_pedido'] == 1082494)].head(1)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_recusada_item_fulfillment,qt_recusada_item_pedido,dt_movimentacao_pedido,rn


In [ ]:
df_recusada30[(df_recusada30['cod_sku'] == '52171') &
              (df_recusada30['cod_pedido'] == 1023802)].head(1)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,qt_recusada_item_fulfillment,qt_recusada_item_pedido,dt_movimentacao_pedido,rn


In [ ]:
df_recusada30 = df_recusada30[
    (df_recusada30['status_item_fulfillment'].isin(['REFUSED'])) &
    #(df_cancelado_franchisee['sequence'] == 1) &
    (df_recusada30['qt_recusada_item_fulfillment'] > 0)
].copy()

In [ ]:
df_recusada30 = df_recusada30.loc[df_recusada30.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmax()].copy()

In [ ]:
df_recusada30['dt_movimentacao_pedido'] = pd.to_datetime(df_recusada30['dt_movimentacao_pedido'])
# Extrair o ano
df_recusada30['ano_movimentacao'] = df_recusada30['dt_movimentacao_pedido'].dt.year
# Extrair o mês
df_recusada30['mes_movimentacao'] = df_recusada30['dt_movimentacao_pedido'].dt.month
# Extrair a semana ISO
df_recusada30['semana_iso_movimentacao'] = df_recusada30['dt_movimentacao_pedido'].dt.isocalendar().week

In [ ]:
df_recusada30 = df_recusada30.merge(df_atendimento_prevista, how='left', left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                              right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
df_recusada30.dtypes

,0
cod_pedido,Int64
cod_item_pedido,Int64
cod_pedido_mar,object
cod_fulfillment,Int64
cod_item_fulfillment,Int64
status_item_fulfillment,object
status_item_fulfillment_interno,object
cod_pdv_ecc_s4,object
cod_sku,object
un_sku,object


In [ ]:
df_recusada30[(df_recusada30['cod_sku'] == 52171) &
              (df_recusada30['cod_pedido'] == 1023802)].head(50)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,status_item_fulfillment,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,...,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


In [ ]:
# nome_arquivo = f'06_0_df_recusada30.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_recusada30.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

## 07.1 - Recusado 30 dias - Vencido

In [ ]:
# Selecionando as colunas
df_vencido = df_recusada30[['cod_pedido', 'cod_item_fulfillment', 'dt_criacao_pedido', 'cod_sku']].copy()

In [ ]:
# Remover linhas com valores nulos
df_vencido = df_vencido.dropna()

In [ ]:
# Remover duplicados com base nas colunas cod_pedido e dt_atendimento_prevista
df_vencido = df_vencido.drop_duplicates(subset=['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
# Ordenar o DataFrame com base na coluna dt_atendimento_prevista
df_vencido = df_vencido.sort_values(by='cod_pedido')

In [ ]:
# Resetar o índice
df_vencido = df_vencido.reset_index(drop=True)

In [ ]:
#df_vencido['obs'] = 'Canceldo'

In [ ]:
# nome_arquivo = f'06_1_df_atendimento_prevista.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_vencido.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

# 08.0 - Recusado manual

In [ ]:
df_recusada = df_pedidos_full[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar', 'cod_fulfillment', 'cod_item_fulfillment', 'status_item_fulfillment', 'status_item_pedido', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4',
                                 'cod_sku', 'un_sku', 'qt_solicitada', 'dt_movimentacao_pedido', 'rn']].copy()

In [ ]:
df_recusada = df_recusada[
    (df_recusada['status_item_pedido'].isin(["REFUSED", "CLOSED"])) &
    (df_recusada['qt_solicitada'] > 0)
].copy()

In [ ]:
df_recusada = df_recusada.loc[df_recusada.groupby(['cod_pedido', 'cod_pdv_ecc_s4', 'cod_sku', 'cod_item_fulfillment'])['rn'].idxmax()].copy()

In [ ]:
df_recusada['dt_movimentacao_pedido'] = pd.to_datetime(df_recusada['dt_movimentacao_pedido'])
# Extrair o ano
df_recusada['ano_movimentacao'] = df_recusada['dt_movimentacao_pedido'].dt.year
# Extrair o mês
df_recusada['mes_movimentacao'] = df_recusada['dt_movimentacao_pedido'].dt.month
# Extrair a semana ISO
df_recusada['semana_iso_movimentacao'] = df_recusada['dt_movimentacao_pedido'].dt.isocalendar().week

In [ ]:
df_recusada = df_recusada.merge(df_atendimento_prevista, how='left', left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                              right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
df_recusada[(df_recusada['cod_sku'] == 51087) &
              (df_recusada['cod_pedido'] == 1019317)].head(50)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_fulfillment,cod_item_fulfillment,status_item_fulfillment,status_item_pedido,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,...,dt_movimentacao_pedido,rn,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao,dt_criacao_pedido,dt_atendimento_prevista,ano_pedido,mes_pedido,semana_iso_pedido


In [ ]:
# nome_arquivo = f'07_0_df_recusada.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_recusada.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

## 08.1 - Cancelado

In [ ]:
# Selecionando as colunas
df_recusada_manual = df_recusada[['cod_pedido', 'cod_item_fulfillment', 'dt_criacao_pedido', 'cod_sku']].copy()

In [ ]:
# Remover linhas com valores nulos
df_recusada_manual = df_recusada_manual.dropna()

In [ ]:
# Remover duplicados com base nas colunas cod_pedido e dt_atendimento_prevista
df_recusada_manual = df_recusada_manual.drop_duplicates(subset=['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
# Ordenar o DataFrame com base na coluna dt_atendimento_prevista
df_recusada_manual = df_recusada_manual.sort_values(by='cod_pedido')

In [ ]:
# Resetar o índice
df_recusada_manual = df_recusada_manual.reset_index(drop=True)

In [ ]:
#df_recusada_manual['obs'] = 'Canceldo manualmente'

In [ ]:
# nome_arquivo = f'07_1_df_recusada_manual.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_recusada_manual.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

# 09.0 Sell in base consolidado

In [ ]:
df_pedidos_fim = df_pedidos[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                         'qt_solicitada',
                         'dt_movimentacao_pedido', 'dt_atendimento_prevista', 'dt_criacao_pedido', 'rn','ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'ano_movimentacao','mes_movimentacao', 'semana_iso_movimentacao']].copy()

In [ ]:
df_cancelado_franchisee_fim = df_cancelado_franchisee[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                                                   'qt_cancelado_franchisee',
                                                   'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn', 'ano_movimentacao','mes_movimentacao', 'semana_iso_movimentacao','dt_atendimento_prevista', 'ano_pedido', 'mes_pedido','semana_iso_pedido']].copy()

In [ ]:
df_phased_fim = df_phased[[ 'cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                        'qt_phased',
                        'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn','ano_movimentacao', 'mes_movimentacao', 'semana_iso_movimentacao','dt_atendimento_prevista', 'ano_pedido', 'mes_pedido','semana_iso_pedido']].copy()

In [ ]:
df_fornecido_fim = df_fornecido[[ 'cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                              'qt_fornecida_item_fulfillment',
                              'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn', 'ano_movimentacao','mes_movimentacao', 'semana_iso_movimentacao','dt_atendimento_prevista', 'ano_pedido', 'mes_pedido','semana_iso_pedido']].copy()

In [ ]:
df_pendente_fim = df_pendente[[ 'cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                              'qt_pendente_item_fulfillment',
                              'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn', 'ano_movimentacao','mes_movimentacao', 'semana_iso_movimentacao','dt_atendimento_prevista', 'ano_pedido', 'mes_pedido','semana_iso_pedido']].copy()

In [ ]:
df_recusada30_fim = df_recusada30.rename(columns={'qt_recusada_item_fulfillment':'qt_recusada_30'})

In [ ]:
df_recusada30_fim = df_recusada30_fim[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                               'qt_recusada_30',
                               'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn', 'ano_movimentacao','mes_movimentacao', 'semana_iso_movimentacao','dt_atendimento_prevista', 'ano_pedido', 'mes_pedido','semana_iso_pedido']].copy()

In [ ]:
df_recusada = df_recusada.rename(columns={'qt_solicitada':'qt_recusada'})

In [ ]:
df_recusada_fim = df_recusada[[ 'cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_pedido', 'status_item_fulfillment_interno','cod_pdv_ecc_s4', 'cod_sku', 'un_sku',
                            'qt_recusada',
                            'dt_movimentacao_pedido', 'dt_criacao_pedido', 'rn', 'ano_movimentacao','mes_movimentacao', 'semana_iso_movimentacao','dt_atendimento_prevista', 'ano_pedido', 'mes_pedido','semana_iso_pedido']].copy()

In [ ]:
df_sell_in = pd.concat([df_pedidos_fim, df_cancelado_franchisee_fim, df_phased_fim, df_fornecido_fim, df_pendente_fim, df_recusada30_fim, df_recusada_fim])

In [ ]:
df_sell_in = df_sell_in[['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar','cod_item_fulfillment', 'status_item_fulfillment', 'status_item_pedido', 'status_item_fulfillment_interno', 'cod_pdv_ecc_s4','cod_sku', 'un_sku',
                         'qt_solicitada', 'qt_cancelado_franchisee', 'qt_phased', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_30', 'qt_recusada', 'dt_criacao_pedido',
                         'dt_movimentacao_pedido', 'ano_pedido', 'mes_pedido','semana_iso_pedido','dt_atendimento_prevista', 'ano_movimentacao', 'mes_movimentacao','semana_iso_movimentacao']].copy()

In [ ]:
df_sell_in = df_sell_in.reset_index(drop=True)

In [ ]:
nomes_colunas = df_sell_in.columns
print(nomes_colunas)

Index(['cod_pedido', 'cod_item_pedido', 'cod_pedido_mar',
       'cod_item_fulfillment', 'status_item_fulfillment', 'status_item_pedido',
       'status_item_fulfillment_interno', 'cod_pdv_ecc_s4', 'cod_sku',
       'un_sku', 'qt_solicitada', 'qt_cancelado_franchisee', 'qt_phased',
       'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment',
       'qt_recusada_30', 'qt_recusada', 'dt_criacao_pedido',
       'dt_movimentacao_pedido', 'ano_pedido', 'mes_pedido',
       'semana_iso_pedido', 'dt_atendimento_prevista', 'ano_movimentacao',
       'mes_movimentacao', 'semana_iso_movimentacao'],
      dtype='object')


In [ ]:
df_sell_in[(df_sell_in['cod_sku'] == 57550) &
           (df_sell_in['cod_pedido'] == 1083253)].head(50)

,cod_pedido,cod_item_pedido,cod_pedido_mar,cod_item_fulfillment,status_item_fulfillment,status_item_pedido,status_item_fulfillment_interno,cod_pdv_ecc_s4,cod_sku,un_sku,...,qt_recusada,dt_criacao_pedido,dt_movimentacao_pedido,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,ano_movimentacao,mes_movimentacao,semana_iso_movimentacao


In [ ]:
df_sell_in_consolidado = df_sell_in[['cod_pedido','cod_pedido_mar', 'cod_item_fulfillment', 'cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'dt_atendimento_prevista', 'dt_criacao_pedido',
                         'qt_solicitada', 'qt_cancelado_franchisee', 'qt_phased', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_30', 'qt_recusada']].copy()

In [ ]:
df_sell_in_consolidado = df_sell_in_consolidado.groupby(
    ['cod_pedido','cod_pedido_mar', 'cod_item_fulfillment', 'cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'dt_atendimento_prevista', 'dt_criacao_pedido']
)[
    ['qt_solicitada', 'qt_cancelado_franchisee', 'qt_phased', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_30', 'qt_recusada']
].sum().reset_index().copy()

In [ ]:
nome_arquivo = f'08_0_df_sell_in.csv'
nome_arquivo_saida = f'{nome_arquivo}'
sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
df_sell_in.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

## 09.1 Ajustando qt_original

In [ ]:
df_sell_in_consolidado[(df_sell_in_consolidado['cod_pedido'] == 1183563) &
              (df_sell_in_consolidado['cod_sku'] == '51447')].head(50)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,dt_criacao_pedido,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada


In [ ]:
df_sell_in_consolidado[(df_sell_in_consolidado['cod_pedido'] == 1183563) &
              (df_sell_in_consolidado['cod_sku'] == 30203)].head(50)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,dt_criacao_pedido,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada


In [ ]:
#df_sell_in_consolidado['qt_original_item_fulfillment'] = df_sell_in_consolidado['qt_solicitada'] - (df_sell_in_consolidado['qt_cancelado_franchisee'] + df_sell_in_consolidado['qt_phased'] - df_sell_in_consolidado['qt_fornecida_item_fulfillment'])
#df_sell_in_consolidado['qt_original_item_fulfillment'] = df_sell_in_consolidado['qt_solicitada'] - df_sell_in_consolidado['qt_cancelado_franchisee'] - df_sell_in_consolidado['qt_phased']
#df_sell_in_consolidado['qt_original_item_fulfillment'] = (df_sell_in_consolidado['qt_solicitada'] - df_sell_in_consolidado['qt_fornecida_item_fulfillment']) - (df_sell_in_consolidado['qt_cancelado_franchisee'] + df_sell_in_consolidado['qt_phased'])

df_sell_in_consolidado['qt_original_item_fulfillment'] = df_sell_in_consolidado['qt_solicitada'] - np.where(
    (df_sell_in_consolidado['qt_cancelado_franchisee'] + df_sell_in_consolidado['qt_phased']) >
    (df_sell_in_consolidado['qt_solicitada'] - df_sell_in_consolidado['qt_fornecida_item_fulfillment'] - df_sell_in_consolidado['qt_pendente_item_fulfillment']  - df_sell_in_consolidado['qt_recusada_30']) ,
    (df_sell_in_consolidado['qt_solicitada'] - df_sell_in_consolidado['qt_fornecida_item_fulfillment'] - df_sell_in_consolidado['qt_pendente_item_fulfillment']  - df_sell_in_consolidado['qt_recusada_30']) ,
    (df_sell_in_consolidado['qt_cancelado_franchisee'] + df_sell_in_consolidado['qt_phased'])
)

In [ ]:
data_atual_fun = pd.to_datetime(data_atual_string)

In [ ]:
# Criando a coluna 'indicador'
df_sell_in_consolidado['indicador'] = df_sell_in_consolidado['dt_atendimento_prevista'].apply(lambda x: 'sim' if x < data_atual else 'não')

In [ ]:
# nome_arquivo = f'08_1_df_sell_in_pedido.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_sell_in_consolidado.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

## 09.2 Validando qt Original

In [ ]:
df_pedidos_full_validacao_ori.head(3)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_pedido
0,3405794,90083,2026-08-26,65
1,3410786,92859,2026-08-27,90
2,3411124,92859,2026-08-27,46


In [ ]:
df_pedidos_full_validacao_sol = df_sell_in_consolidado[['cod_pedido', 'cod_sku', 'dt_criacao_pedido', 'qt_original_item_fulfillment', 'qt_solicitada',
                                                        'qt_cancelado_franchisee', 'qt_phased', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_30', 'qt_recusada',]].copy()

In [ ]:
df_pedidos_full_validacao_sol[(df_pedidos_full_validacao_sol['cod_pedido'] == 1183563) &
              (df_pedidos_full_validacao_sol['cod_sku'] == '51447')].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_fulfillment,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada


In [ ]:
df_pedidos_full_validacao_ori[(df_pedidos_full_validacao_ori['cod_pedido'] == 1183563) &
              (df_pedidos_full_validacao_ori['cod_sku'] == '51447')].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_pedido


In [ ]:
df_pedidos_full_validacao_sol = df_pedidos_full_validacao_sol.groupby(
    ['cod_pedido','cod_sku', 'dt_criacao_pedido']
)[
    ['qt_original_item_fulfillment', 'qt_solicitada', 'qt_cancelado_franchisee', 'qt_phased', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_30', 'qt_recusada']
].sum().reset_index().copy()

In [ ]:
df_pedidos_full_validacao_sol[(df_pedidos_full_validacao_sol['cod_pedido'] == 1183563) &
              (df_pedidos_full_validacao_sol['cod_sku'] == 51447)].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_fulfillment,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada


In [ ]:
df_pedidos_full_validacao_ori[(df_pedidos_full_validacao_ori['cod_pedido'] == 1183563) &
              (df_pedidos_full_validacao_ori['cod_sku'] == 51447)].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_pedido


In [ ]:
df_pedidos_full_validacao_sol.rename(columns={'qt_original_item_fulfillment':'qt_original_item_fulfillment_ajustado'}, inplace=True)

In [ ]:
df_pedidos_full_validacao = pd.concat([df_pedidos_full_validacao_ori, df_pedidos_full_validacao_sol])

In [ ]:
df_pedidos_full_validacao = df_pedidos_full_validacao.groupby(
    ['cod_pedido','cod_sku', 'dt_criacao_pedido']
)[
    ['qt_original_item_pedido', 'qt_original_item_fulfillment_ajustado', 'qt_solicitada', 'qt_cancelado_franchisee', 'qt_phased', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_recusada_30', 'qt_recusada']
].sum().reset_index().copy()

In [ ]:
df_pedidos_full_validacao_ori[(df_pedidos_full_validacao_ori['cod_pedido'] == 1099235) &
              (df_pedidos_full_validacao_ori['cod_sku'] == 59958)].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_pedido


In [ ]:
df_pedidos_full_validacao['delta'] = df_pedidos_full_validacao['qt_original_item_fulfillment_ajustado'] - df_pedidos_full_validacao['qt_original_item_pedido']
df_pedidos_full_validacao['obs'] = np.where(df_pedidos_full_validacao['delta'] != 0, 'Erro', 'OK')

In [ ]:
# nome_arquivo = f'99_9_df_sell_in_erro.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_pedidos_full_validacao.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_pedidos_full_validacao_ori[(df_pedidos_full_validacao_ori['cod_pedido'] == 1099235) &
              (df_pedidos_full_validacao_ori['cod_sku'] == 59958)].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_pedido


In [ ]:
df_pedidos_full_validacao_sol[(df_pedidos_full_validacao_sol['cod_pedido'] == 1022180) &
              (df_pedidos_full_validacao_sol['cod_sku'] == 53434)].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_fulfillment_ajustado,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada


In [ ]:
df_pedidos_full_validacao[(df_pedidos_full_validacao['cod_pedido'] == 1022180) &
              (df_pedidos_full_validacao['cod_sku'] == 53434)].head(50)

,cod_pedido,cod_sku,dt_criacao_pedido,qt_original_item_pedido,qt_original_item_fulfillment_ajustado,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada,delta,obs


In [ ]:
df_sell_in_consolidado[(df_sell_in_consolidado['cod_pedido'] == 991879) &
              (df_sell_in_consolidado['cod_sku'] == 30203)].head(50)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,dt_criacao_pedido,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada,qt_original_item_fulfillment,indicador


In [ ]:
df_sell_in_consolidado[(df_sell_in_consolidado['cod_pedido'] == 991879) &
              (df_sell_in_consolidado['cod_sku'] == 30203)].head(50)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,dt_criacao_pedido,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada,qt_original_item_fulfillment,indicador


In [ ]:
df_sell_in_consolidado.head(3)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,dt_criacao_pedido,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada,qt_original_item_fulfillment,indicador
0,3196859,2305411,1575245662,2100003142,94664,2026.0,9.0,39,2026-09-22,2026-07-09,0,0,0,70,0,0,0,70,sim
1,3196859,2305411,1575245955,2100003142,94665,2026.0,9.0,39,2026-09-22,2026-07-09,0,0,0,20,0,0,0,20,sim
2,3196859,2305411,1575245958,2100003142,94666,2026.0,9.0,39,2026-09-22,2026-07-09,0,0,0,21,0,0,0,21,sim


# 09.0 Cancelados consolidados

In [ ]:
df_sell_in_cancelados = pd.concat([df_vencido, df_recusada_manual])

In [ ]:
# Remover linhas com valores nulos
df_sell_in_cancelados = df_sell_in_cancelados.dropna()

In [ ]:
# Remover duplicados com base nas colunas cod_pedido e dt_atendimento_prevista
df_sell_in_cancelados = df_sell_in_cancelados.drop_duplicates(subset=['cod_pedido', 'cod_sku'])

In [ ]:
# Ordenar o DataFrame com base na coluna dt_atendimento_prevista
df_sell_in_cancelados = df_sell_in_cancelados.sort_values(by='cod_pedido')

In [ ]:
# Resetar o índice
df_sell_in_cancelados = df_sell_in_cancelados.reset_index(drop=True)

In [ ]:
df_sell_in_cancelados['obs'] = 'Cancelado'

In [ ]:
df_sell_in_cancelados.head(3)

,cod_pedido,cod_item_fulfillment,dt_criacao_pedido,cod_sku,obs
0,3197590,1575114229,2026-07-09,94661,Cancelado
1,3197590,1575114208,2026-07-09,94666,Cancelado
2,3197590,1575114217,2026-07-09,94669,Cancelado


# 10.0 Sem duplicados

In [ ]:
df_sell_in_smart = df_sell_in_consolidado.copy()

In [ ]:
df_sell_in_smart.head(3)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,dt_atendimento_prevista,dt_criacao_pedido,qt_solicitada,qt_cancelado_franchisee,qt_phased,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_recusada_30,qt_recusada,qt_original_item_fulfillment,indicador
0,3196859,2305411,1575245662,2100003142,94664,2026.0,9.0,39,2026-09-22,2026-07-09,0,0,0,70,0,0,0,70,sim
1,3196859,2305411,1575245955,2100003142,94665,2026.0,9.0,39,2026-09-22,2026-07-09,0,0,0,20,0,0,0,20,sim
2,3196859,2305411,1575245958,2100003142,94666,2026.0,9.0,39,2026-09-22,2026-07-09,0,0,0,21,0,0,0,21,sim


In [ ]:
df_sell_in_smart = df_sell_in_smart[['cod_pedido','cod_pedido_mar', 'cod_item_fulfillment', 'cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'indicador', 'qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment']]

In [ ]:
df_sell_in_smart = df_sell_in_smart.groupby(
    ['cod_pedido','cod_pedido_mar', 'cod_item_fulfillment', 'cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'indicador']
)[
    ['qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment']
].sum().reset_index().copy()

In [ ]:
df_sell_in_smart = df_sell_in_smart.merge(df_sell_in_cancelados, how='left', left_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'],
                              right_on = ['cod_pedido', 'cod_item_fulfillment', 'cod_sku'])

In [ ]:
df_sell_in_smart.head(3)

,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,indicador,qt_original_item_fulfillment,qt_fornecida_item_fulfillment,dt_criacao_pedido,obs
0,3196859,2305411,1575245662,2100003142,94664,2026.0,9.0,39,sim,70,70,NaN,NaN
1,3196859,2305411,1575245955,2100003142,94665,2026.0,9.0,39,sim,20,20,NaN,NaN
2,3196859,2305411,1575245958,2100003142,94666,2026.0,9.0,39,sim,21,21,NaN,NaN


In [ ]:
df_sell_in_smart.query('cod_item_fulfillment == 1513755184').head(2)

/tmp/ipykernel_1592/2806139844.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  df_sell_in_smart.query('cod_item_fulfillment == 1513755184').head(2)


,cod_pedido,cod_pedido_mar,cod_item_fulfillment,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,indicador,qt_original_item_fulfillment,qt_fornecida_item_fulfillment,dt_criacao_pedido,obs


In [ ]:
df_sell_in_smart = df_sell_in_smart.merge(df_pdvs, how='left', left_on = ['cod_pdv_ecc_s4'],
                              right_on = ['cod_de_para'])


In [ ]:
#df_sell_in_smart = df_sell_in_smart.loc[(df_sell_in_smart['cod_canal'] == 'VDF_FRANQ')]

In [ ]:
df_sell_in_smart = df_sell_in_smart.loc[(df_sell_in_smart['ano_pedido'] == 2026)]

In [ ]:
# nome_arquivo = f'10_0_df_sell_in_pre_semanal.csv'
# nome_arquivo_saida = f'{nome_arquivo}'
# sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
# pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
# df_sell_in_smart.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

## 10.1 Classificando pedidos

In [ ]:
df_sell_in_smart['status'] = np.where(
    ((df_sell_in_smart['qt_original_item_fulfillment'] - df_sell_in_smart['qt_fornecida_item_fulfillment'] <= 0) | (df_sell_in_smart['obs'] == 'Cancelado')),
    'Encerrado',
    'Aberto'
)

In [ ]:
df_sell_in_smart['qt_pendente_item_fulfillment'] = np.where(
    df_sell_in_smart['status'] == 'Aberto',
    df_sell_in_smart['qt_original_item_fulfillment'] - df_sell_in_smart['qt_fornecida_item_fulfillment'],
    np.where(
        df_sell_in_smart['status'] == 'Encerrado',
        0,  # Se o status for 'Encerrado', a quantidade pendente é 0
        np.nan  # Se o status for outro valor, atribuir NaN (outra condição não especificada)
    )
)

In [ ]:
df_sell_in_smart['qt_cancelada_item_fulfillment'] = np.where(
    df_sell_in_smart['status'] == 'Encerrado',
    df_sell_in_smart['qt_original_item_fulfillment'] - df_sell_in_smart['qt_fornecida_item_fulfillment'] - df_sell_in_smart['qt_pendente_item_fulfillment'],
    np.where(
        df_sell_in_smart['status'] == 'Aberto',
        0,  # Se o status for 'Aberto', a quantidade cancelada é 0
        np.nan  # Se o status for outro valor, atribuir NaN (outra condição não especificada)
    )
)

In [ ]:
df_sell_in_smart = df_sell_in_smart[['cod_pedido','cod_pedido_mar', 'cod_item_fulfillment', 'cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'indicador','status', 'qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_cancelada_item_fulfillment']]

In [ ]:
 nome_arquivo = f'10_1_df_sell_in_pedido_semanal_classificando_pedido.csv'
 nome_arquivo_saida = f'{nome_arquivo}'
 sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
 pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
 df_sell_in_smart.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

## 10.2 Semanal com demanda estimada

In [ ]:
df_sell_in_smart_demanda = df_sell_in_smart.groupby(
    ['cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'indicador']
)[
    ['qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_cancelada_item_fulfillment']
].sum().reset_index().copy()

In [ ]:
df_sell_in_smart_demanda[(df_sell_in_smart_demanda['cod_pdv_ecc_s4'] == '2100000655')].query('cod_sku == "01641"')

,cod_pdv_ecc_s4,cod_sku,ano_pedido,mes_pedido,semana_iso_pedido,indicador,qt_original_item_fulfillment,qt_fornecida_item_fulfillment,qt_pendente_item_fulfillment,qt_cancelada_item_fulfillment
467489,2100000655,01641,2026.0,9.0,40,sim,43,43,0.0,0.0
467490,2100000655,01641,2026.0,10.0,41,sim,73,0,0.0,73.0
467491,2100000655,01641,2026.0,10.0,42,sim,90,0,0.0,90.0


In [ ]:
# Ordenar o DataFrame por 'cod_pdv_ecc', 'cod_sku' e 'semana_iso_pedido'
df_sell_in_smart_demanda.sort_values(by=['cod_pdv_ecc_s4', 'cod_sku', 'mes_pedido','semana_iso_pedido', 'indicador'], inplace=True)

In [ ]:
# Resetar o índice
df_sell_in_smart_demanda.reset_index(drop=True, inplace=True)

In [ ]:
# Calcular a coluna 'qt_pendente_dem_item_fulfillment'
df_sell_in_smart_demanda['qt_pendente_dem_item_fulfillment'] = df_sell_in_smart_demanda['qt_original_item_fulfillment'] - df_sell_in_smart_demanda['qt_fornecida_item_fulfillment']
df_sell_in_smart_demanda['qt_pendente_dem_item_fulfillment'] = df_sell_in_smart_demanda['qt_pendente_dem_item_fulfillment'].apply(lambda x: max(0, x))

In [ ]:
# Calcular a coluna 'qt_demanda_estimada'
df_sell_in_smart_demanda['qt_demanda_estimada'] = df_sell_in_smart_demanda['qt_original_item_fulfillment']  # Inicialize com o valor da coluna 'qt_original_item_fulfillment'

In [ ]:
for i in range(1, len(df_sell_in_smart_demanda)):
    # Verificar se os campos da linha atual são iguais aos da linha acima
    if (df_sell_in_smart_demanda.at[i, 'cod_pdv_ecc_s4'] == df_sell_in_smart_demanda.at[i-1, 'cod_pdv_ecc_s4'] and
        df_sell_in_smart_demanda.at[i, 'cod_sku'] == df_sell_in_smart_demanda.at[i-1, 'cod_sku'] and
        df_sell_in_smart_demanda.at[i, 'mes_pedido'] == df_sell_in_smart_demanda.at[i-1, 'mes_pedido'] and
        df_sell_in_smart_demanda.at[i, 'semana_iso_pedido'] - df_sell_in_smart_demanda.at[i-1, 'semana_iso_pedido'] == 1):

        # Calcular qt_demanda_estimada com base nos valores da linha atual e da linha acima
        qt_demand_above = df_sell_in_smart_demanda.at[i-1, 'qt_pendente_dem_item_fulfillment']
        qt_original_current = df_sell_in_smart_demanda.at[i, 'qt_original_item_fulfillment']
        df_sell_in_smart_demanda.at[i, 'qt_demanda_estimada'] = max(0, qt_original_current - qt_demand_above)

In [ ]:
# Ajustar os valores negativos para zero
df_sell_in_smart_demanda['qt_demanda_estimada'] = df_sell_in_smart_demanda['qt_demanda_estimada'].apply(lambda x: max(0, x))

In [ ]:

df_sell_in_smart_demanda = df_sell_in_smart_demanda.merge(df_pdvs, how='left', left_on = ['cod_pdv_ecc_s4'],
                              right_on = ['cod_de_para'])

In [ ]:
# Convertendo uma coluna em valores inteiros usando pd.to_numeric()
df_sell_in_smart_demanda['cod_sku'] = pd.to_numeric(df_sell_in_smart_demanda['cod_sku'], downcast='integer', errors='coerce')

In [ ]:
df_sku = df_sku.rename(columns={'cod_material':'cod_sku'}).copy()

In [ ]:
df_sell_in_smart_demanda = df_sell_in_smart_demanda.merge(df_sku, how='left', left_on = ['cod_sku'],
                              right_on = ['cod_sku'])

In [ ]:
df_sell_in_smart_demanda['semana_iso_pedido'] = pd.to_numeric(df_sell_in_smart_demanda['semana_iso_pedido'], downcast='integer', errors='coerce')
df_sell_in_smart_demanda['ano_pedido'] = pd.to_numeric(df_sell_in_smart_demanda['ano_pedido'], downcast='integer', errors='coerce')
df_sell_in_smart_demanda['cod_sku'] = pd.to_numeric(df_sell_in_smart_demanda['cod_sku'], downcast='integer', errors='coerce')

In [ ]:
df_sell_in_smart_demanda = df_sell_in_smart_demanda.merge(df_02_8_itens_criticos, how='left', left_on = ['cod_cd', 'semana_iso_pedido', 'ano_pedido', 'cod_sku'],
                              right_on = ['cod_cd', 'semana_iso_pedido', 'ano_pedido', 'cod_sku'])

In [ ]:
df_sell_in_smart_demanda['Crítico'].fillna("Nâo", inplace=True)

/tmp/ipykernel_1592/3718092317.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_sell_in_smart_demanda['Crítico'].fillna("Nâo", inplace=True)


In [ ]:
df_sell_in_smart_demanda['qt_nao_atendida'] = df_sell_in_smart_demanda['qt_original_item_fulfillment'] - df_sell_in_smart_demanda['qt_fornecida_item_fulfillment']
df_sell_in_smart_demanda['qt_nao_atendida'] = df_sell_in_smart_demanda['qt_nao_atendida'].apply(lambda x: max(0, x))

In [ ]:
df_sell_in_smart_demanda['qt_fornecida_item_fulfillment_dem_estimada'] = df_sell_in_smart_demanda['qt_fornecida_item_fulfillment'].apply(formatar_numero)

In [ ]:
df_sell_in_smart_demanda['qt_fornecida_item_fulfillment_dem_estimada'] = df_sell_in_smart_demanda['qt_fornecida_item_fulfillment_dem_estimada'].replace('nan', np.nan).astype(float)

In [ ]:
df_sell_in_smart_demanda['qt_nao_atendida_dem_estimada'] = df_sell_in_smart_demanda['qt_demanda_estimada'] - df_sell_in_smart_demanda['qt_fornecida_item_fulfillment']
df_sell_in_smart_demanda['qt_nao_atendida_dem_estimada'] = df_sell_in_smart_demanda['qt_nao_atendida_dem_estimada'].apply(lambda x: max(0, x))

In [ ]:
#nomes_das_colunas = df_completo.columns
print(df_sell_in_smart_demanda.columns)

Index(['cod_pdv_ecc_s4', 'cod_sku', 'ano_pedido', 'mes_pedido',
       'semana_iso_pedido', 'indicador', 'qt_original_item_fulfillment',
       'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment',
       'qt_cancelada_item_fulfillment', 'qt_pendente_dem_item_fulfillment',
       'qt_demanda_estimada', 'cod_pdv', 'cod_de_para', 'cod_cliente_s4',
       'cod_cd', 'cod_uf', 'cod_canal', 'des_material', 'cod_un_material',
       'cod_marca_material', 'des_marca_material', 'cod_categoria_material',
       'des_categoria_material', 'cod_subcategoria_material',
       'des_subcategoria_material', 'cod_gestao_mercadologica_material',
       'des_gestao_mercadologica_material', 'cod_status_material',
       'des_status_material', 'cod_tipo_material', 'des_tipo_material',
       'cod_hierarquia_material', 'cod_condicao_material',
       'des_condicao_material', 'des_faixa_etaria_material',
       'des_genero_material', 'cod_agrup_sap_material',
       'des_agrup_sap_material', 'cod_a

In [ ]:
df_sell_in_smart_demanda = df_sell_in_smart_demanda[[
      'ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'indicador',
      'cod_pdv_ecc_s4', 'cod_pdv', 'cod_de_para', 'cod_cd', 'cod_uf', 'cod_canal',
      'cod_sku', 'des_material', 'des_marca_material', 'cod_classificacao_abc_material',
      'des_categoria_material', 'des_subcategoria_material', 'des_status_material', 'des_condicao_material', 'Crítico',
      'qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment', 'qt_nao_atendida', 'qt_pendente_item_fulfillment', 'qt_cancelada_item_fulfillment',
      'qt_demanda_estimada', 'qt_fornecida_item_fulfillment_dem_estimada', 'qt_nao_atendida_dem_estimada',
       ]].copy()

In [ ]:
#nomes_das_colunas = df_completo.columns
print(df_sell_in_smart_demanda.columns)

Index(['ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'indicador',
       'cod_pdv_ecc_s4', 'cod_pdv', 'cod_de_para', 'cod_cd', 'cod_uf',
       'cod_canal', 'cod_sku', 'des_material', 'des_marca_material',
       'cod_classificacao_abc_material', 'des_categoria_material',
       'des_subcategoria_material', 'des_status_material',
       'des_condicao_material', 'Crítico', 'qt_original_item_fulfillment',
       'qt_fornecida_item_fulfillment', 'qt_nao_atendida',
       'qt_pendente_item_fulfillment', 'qt_cancelada_item_fulfillment',
       'qt_demanda_estimada', 'qt_fornecida_item_fulfillment_dem_estimada',
       'qt_nao_atendida_dem_estimada'],
      dtype='object')


In [ ]:
df_sell_in_smart_demanda['cod_pdv_ecc_s4'] = df_sell_in_smart_demanda['cod_pdv_ecc_s4'].astype(int)
df_sell_in_smart_demanda['cod_sku'] = df_sell_in_smart_demanda['cod_sku'].astype(int)
df_sell_in_smart_demanda['ano_pedido'] = df_sell_in_smart_demanda['ano_pedido'].astype(int)
df_sell_in_smart_demanda['mes_pedido'] = df_sell_in_smart_demanda['mes_pedido'].astype(int)
df_sell_in_smart_demanda['semana_iso_pedido'] = df_sell_in_smart_demanda['semana_iso_pedido'].astype(int)

In [ ]:
df_sell_in_smart_demanda = df_sell_in_smart_demanda.merge(df_sugestao_compra_resumo, how='left', left_on = ['ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'cod_pdv_ecc_s4', 'cod_sku'],
                              right_on = ['ano_pedido', 'mes_pedido', 'semana_iso_pedido', 'cod_pdv_ecc_s4', 'cod_sku'])

In [ ]:
df_sell_in_smart_demanda = df_sell_in_smart_demanda.merge(df_preco_calamo, how='left', left_on = ['cod_sku'],
                              right_on = ['cod_sku'])

In [ ]:
nome_arquivo = f'10_2_df_sell_in_semanal.csv'
nome_arquivo_saida = f'{nome_arquivo}'
sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
df_sell_in_smart_demanda.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
df_teste = df_sell_in_smart_demanda[(df_sell_in_smart_demanda['cod_pdv_ecc_s4'] == 12503)]

In [ ]:
df_teste.query('cod_sku == "01641"')

,ano_pedido,mes_pedido,semana_iso_pedido,indicador,cod_pdv_ecc_s4,cod_pdv,cod_de_para,cod_cd,cod_uf,cod_canal,...,qt_fornecida_item_fulfillment_dem_estimada,qt_nao_atendida_dem_estimada,qt_sc_media,qt_sc_minimo,qt_sc_maximo,qt_dias_cobertura_alvo_media,qt_dias_cobertura_alvo_minimo,qt_dias_cobertura_alvo_maximo,qt_dias_ddv_media,vlr_preco_medio


## 10.3

In [ ]:
df_sell_in_smart_demanda_mes = df_sell_in_smart_demanda

In [ ]:
df_sell_in_smart_demanda_mes['cod_pdv_ecc_s4'] = df_sell_in_smart_demanda_mes['cod_pdv']
df_sell_in_smart_demanda_mes['cod_de_para'] = df_sell_in_smart_demanda_mes['cod_pdv']

In [ ]:
df_sell_in_smart_demanda_mes = df_sell_in_smart_demanda_mes.groupby(
    ['ano_pedido', 'mes_pedido', 'indicador',
      'cod_pdv_ecc_s4', 'cod_pdv', 'cod_de_para', 'cod_cd', 'cod_uf', 'cod_canal',
      'cod_sku', 'des_material', 'des_marca_material', 'cod_classificacao_abc_material',
      'des_categoria_material', 'des_subcategoria_material', 'des_status_material', 'des_condicao_material']
)[
    ['qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_cancelada_item_fulfillment',
      'qt_demanda_estimada', 'qt_fornecida_item_fulfillment_dem_estimada']
].sum().reset_index().copy()

In [ ]:
# Transformar a coluna em float
df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada'] = df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada'].replace('nan', np.nan).astype(float)

In [ ]:
df_sell_in_smart_demanda_mes = df_sell_in_smart_demanda_mes[[
      'ano_pedido', 'mes_pedido', 'indicador',
      'cod_pdv_ecc_s4', 'cod_pdv', 'cod_de_para', 'cod_cd', 'cod_uf', 'cod_canal',
      'cod_sku', 'des_material', 'des_marca_material', 'cod_classificacao_abc_material',
      'des_categoria_material', 'des_subcategoria_material', 'des_status_material', 'des_condicao_material',
      'qt_original_item_fulfillment', 'qt_fornecida_item_fulfillment', 'qt_pendente_item_fulfillment', 'qt_cancelada_item_fulfillment',
      'qt_demanda_estimada', 'qt_fornecida_item_fulfillment_dem_estimada',
       ]].copy()

In [ ]:
# Definir as condições
conditions_supply = (
    (df_sell_in_smart_demanda_mes['des_status_material'] == 'Cancelado') |
    (df_sell_in_smart_demanda_mes['des_status_material'] == 'Desativado') |
    (df_sell_in_smart_demanda_mes['des_status_material'] == 'Desativado com Estoque') |
    (df_sell_in_smart_demanda_mes['des_status_material'] == 'Lançamento Fluxo Incompl.') |
    (df_sell_in_smart_demanda_mes['des_status_material'] == 'Pré-Cadastro') |
    (df_sell_in_smart_demanda_mes['des_status_material'] == 'Somente Controle Interno') |
    (df_sell_in_smart_demanda_mes['des_condicao_material'] == 'EDICAO LIMITADA') |
    (df_sell_in_smart_demanda_mes['des_categoria_material'] == 'SUPORTE À VENDA') |
    (df_sell_in_smart_demanda_mes['des_categoria_material'] == 'EMBALAGENS') |
    (df_sell_in_smart_demanda_mes['des_categoria_material'] == 'ACESSÓRIOS (Desativado)') |
    (df_sell_in_smart_demanda_mes['des_categoria_material'] == 'MAQUIAGEM (Desativado)') |
    (df_sell_in_smart_demanda_mes['des_categoria_material'] == 'DESATIVADA')
    # (df_sell_in_smart_demanda_mes['vlr_compra_re'] == 0) |
    # (df_sell_in_smart_demanda_mes['pv_unit_mes'] <= 0)
)

In [ ]:
# Atualizar a coluna 'qt_demanda_estimada' para ser igual a 'qt_fornecida_item_fulfillment_dem_estimada' quando as condições são verdadeiras
df_sell_in_smart_demanda_mes['qt_demanda_estimada'] = np.where(
    conditions_supply,
    df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada'],
    df_sell_in_smart_demanda_mes['qt_demanda_estimada']
)

In [ ]:
df_sell_in_smart_demanda_mes['qt_nao_atendida'] = df_sell_in_smart_demanda_mes['qt_original_item_fulfillment'] - df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment']
df_sell_in_smart_demanda_mes['qt_nao_atendida'] = df_sell_in_smart_demanda_mes['qt_nao_atendida'].apply(lambda x: max(0, x))

In [ ]:
df_sell_in_smart_demanda_mes['qt_nao_atendida_dem_estimada'] = df_sell_in_smart_demanda_mes['qt_demanda_estimada'] - df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada']
df_sell_in_smart_demanda_mes['qt_nao_atendida_dem_estimada'] = df_sell_in_smart_demanda_mes['qt_nao_atendida_dem_estimada'].apply(lambda x: max(0, x))

In [ ]:
df_sell_in_smart_demanda_mes['cod_pdv_ecc_s4'] = df_sell_in_smart_demanda_mes['cod_pdv_ecc_s4'].astype(str)
df_sell_in_smart_demanda_mes['cod_sku'] = df_sell_in_smart_demanda_mes['cod_sku'].astype(int)
df_sell_in_smart_demanda_mes['ano_pedido'] = df_sell_in_smart_demanda_mes['ano_pedido'].astype(int)
df_sell_in_smart_demanda_mes['mes_pedido'] = df_sell_in_smart_demanda_mes['mes_pedido'].astype(int)

In [ ]:
df_sell_in_smart_demanda_mes = df_sell_in_smart_demanda_mes.merge(df_preco_calamo, how='left', left_on = ['cod_sku'],
                              right_on = ['cod_sku'])

In [ ]:
df_sell_in_smart_demanda_mes['vlr_preco_medio'].fillna(0, inplace=True)

/tmp/ipykernel_1592/3129189822.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_sell_in_smart_demanda_mes['vlr_preco_medio'].fillna(0, inplace=True)


In [ ]:
df_sell_in_smart_demanda_mes['pvr_original_item_fulfillment'] = df_sell_in_smart_demanda_mes['qt_original_item_fulfillment'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_fornecida_item_fulfillment'] = df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_nao_atendida'] = df_sell_in_smart_demanda_mes['qt_nao_atendida'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_pendente_item_fulfillment'] = df_sell_in_smart_demanda_mes['qt_pendente_item_fulfillment'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_cancelada_item_fulfillment'] = df_sell_in_smart_demanda_mes['qt_cancelada_item_fulfillment'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_demanda_estimada'] = df_sell_in_smart_demanda_mes['qt_demanda_estimada'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_fornecida_item_fulfillment_dem_estimada'] = df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_nao_atendida_dem_estimada'] = df_sell_in_smart_demanda_mes['qt_nao_atendida_dem_estimada'] * df_sell_in_smart_demanda_mes['vlr_preco_medio']
df_sell_in_smart_demanda_mes['pvr_sc_media'] = 0
df_sell_in_smart_demanda_mes['pvr_sc_minimo'] = 0
df_sell_in_smart_demanda_mes['pvr_sc_maximo'] = 0

In [ ]:
nome_arquivo = f'11_df_sell_in_mes_old_7.csv'
nome_arquivo_saida = f'{nome_arquivo}'
sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
df_sell_in_smart_demanda_mes.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

# 99 Consolidação Arquivos

In [ ]:
full_df = pd.DataFrame(columns = [
    'ano_mes','cod_pdv','cod_de_para','sku_com_de_para','pv_unit_mes','pv_unit_futuro','pv_unit_passado','vendas_unit','vendas_pvr_liquido','qtd_de_dias_com_estoque_disponivel','qtd_de_dias_com_estoque_disponivel_com_vendas','des_material','des_marca_material','des_categoria_material','des_condicao_material','des_status_material','cod_classificacao_abc_material','qt_por_caixa','de_para_agrupado','num_itens','neverstockout','max_lt','des_cadeia_seg_ajustada','des_planejador_mrp','cod_cd','cod_regiao_vendas','cod_uf','cod_canal','cod_tp_loja','cod_logradouro_pais_franquia','cod_logradouro_uf_franquia','des_logradouro_cidade_franquia','des_logradouro_bairro_franquia','cod_logradouro_cep_franquia','alto_lead_time','multi_corte','pvr_unit_uf','pvr_unit_br','ano','mes','data_atual','qtd_total_dias_ciclo','qtd_total_dias_ciclo_passado','qtd_total_dias_ciclo_hoje_futuro','qtd_total_dias_uteis_ciclo','qtd_total_dias_uteis_ciclo_passado','qtd_total_dias_uteis_ciclo_hoje_futuro','qtd_total_dias_que_abastece','progresso_ciclo','possivel_abastecer','pvr_unit','qtd_de_dias_sem_estoque_disponivel_com_vendas','qtd_de_dias_sem_estoque_disponivel','vmd_unit_disponivel_com_vendas','vmd_unit_disponivel','pvr_ref','pv_pvr','pv_pvr_futuro','pv_pvr_passado','vmd_pvr_disponivel_com_vendas','vmd_pvr_disponivel','perdas_unit_disponivel_com_vendas_full','perdas_unit_disponivel_full','pvr_indicador','perdas_pvr_disponivel_com_vendas_full','perdas_pvr_disponivel_full','perdas_unit_mar','perdas_pvr_mar','perdas_unit_customer','perdas_pvr_customer','perdas_unit_supply','perdas_pvr_supply','ponte_supply_pv_unit_passado','ponte_supply_vendas_unit','ponte_supply_perdas_unit','ponte_supply_demandado_unit','ponte_supply_vendas_unit_abaixo','ponte_supply_ruptura_unit_abaixo','ponte_supply_vendas_unit_acima','ponte_supply_ruptura_unit_acima','ponte_supply_pv_pvr_passado','ponte_supply_vendas_pvr','ponte_supply_perdas_pvr','ponte_supply_demandado_pvr','ponte_supply_vendas_pvr_abaixo','ponte_supply_ruptura_pvr_abaixo','ponte_supply_vendas_pvr_acima','ponte_supply_ruptura_pvr_acima','ponte_mar_pv_unit_passado','ponte_mar_vendas_unit','ponte_mar_perdas_unit','ponte_mar_demandado_unit','ponte_mar_vendas_unit_abaixo','ponte_mar_ruptura_unit_abaixo','ponte_mar_vendas_unit_acima','ponte_mar_ruptura_unit_acima','ponte_mar_pv_pvr_passado','ponte_mar_vendas_pvr','ponte_mar_perdas_pvr','ponte_mar_demandado_pvr','ponte_mar_vendas_pvr_abaixo','ponte_mar_ruptura_pvr_abaixo','ponte_mar_vendas_pvr_acima','ponte_mar_ruptura_pvr_acima','ponte_customer_pv_unit_passado','ponte_customer_vendas_unit','ponte_customer_perdas_unit','ponte_customer_demandado_unit','ponte_customer_vendas_unit_abaixo','ponte_customer_ruptura_unit_abaixo','ponte_customer_vendas_unit_acima','ponte_customer_ruptura_unit_acima','ponte_customer_pv_pvr_passado','ponte_customer_vendas_pvr','ponte_customer_perdas_pvr','ponte_customer_demandado_pvr','ponte_customer_vendas_pvr_abaixo','ponte_customer_ruptura_pvr_abaixo','ponte_customer_vendas_pvr_acima','ponte_customer_ruptura_pvr_acima','erro_abs_unit_vendas','erro_abs_unit_customer_demandado','erro_abs_unit_mar_demandado','bias_vendas_unit','wape_vendas_unit','bias_customer_demandado_unit','wape_customer_demandado_unit','bias_mar_demandado_unit','wape_mar_demandado_unit','nivel_de_atedimento_unit_mar','nivel_de_atedimento_unit_supply','nivel_de_atedimento_unit_customer','oportunidade_unit','oportunidade_pvr','td_oportunidade_unit','td_oportunidade_pvr'
])
full_df.loc[len(full_df)] = 0

In [ ]:
# *** ATENÇÃO **** AJUSTAR COM O MÊS ****
df_sell_in_smart_demanda_mes = df_sell_in_smart_demanda_mes[df_sell_in_smart_demanda_mes['mes_pedido'] == int(data_str[5:])]

# Cria um novo DataFrame com as colunas do full_df e os valores do df_sell_in_smart_demanda_mes
new_data = pd.DataFrame({
    'ano_mes': data_str,
    'ano': data_str[:4],
    'mes': data_str[5:],
    'cod_pdv': df_sell_in_smart_demanda_mes['cod_pdv'],
    'cod_de_para': df_sell_in_smart_demanda_mes['cod_pdv'],
    'sku_com_de_para': df_sell_in_smart_demanda_mes['cod_sku'],
    'vendas_unit': df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada'],
    'vendas_pvr_liquido': df_sell_in_smart_demanda_mes['pvr_fornecida_item_fulfillment_dem_estimada'],
    'des_material': df_sell_in_smart_demanda_mes['des_material'],
    'des_marca_material': df_sell_in_smart_demanda_mes['des_marca_material'],
    'des_categoria_material': df_sell_in_smart_demanda_mes['des_categoria_material'],
    'des_condicao_material': df_sell_in_smart_demanda_mes['des_condicao_material'],
    'cod_cd': df_sell_in_smart_demanda_mes['cod_cd'],
    'cod_uf': df_sell_in_smart_demanda_mes['cod_uf'],
    'cod_canal': df_sell_in_smart_demanda_mes['cod_canal'],
    'cod_tp_loja': "LOJA",     # ATENÇÃO - coluna com valor fixo
    'cod_logradouro_pais_franquia': "BR", # ATENÇÃO - coluna com valor fixo
    'perdas_unit_supply': df_sell_in_smart_demanda_mes['qt_nao_atendida_dem_estimada'],
    'perdas_pvr_supply': df_sell_in_smart_demanda_mes['pvr_demanda_estimada'],
    'ponte_supply_vendas_unit': df_sell_in_smart_demanda_mes['qt_fornecida_item_fulfillment_dem_estimada'],
    'ponte_supply_perdas_unit': df_sell_in_smart_demanda_mes['qt_nao_atendida_dem_estimada'],
    'ponte_supply_demandado_unit': df_sell_in_smart_demanda_mes['qt_demanda_estimada'],
    'ponte_supply_vendas_pvr': df_sell_in_smart_demanda_mes['pvr_fornecida_item_fulfillment_dem_estimada'],
    'ponte_supply_perdas_pvr': df_sell_in_smart_demanda_mes['pvr_nao_atendida_dem_estimada'],
    'ponte_supply_demandado_pvr': df_sell_in_smart_demanda_mes['pvr_demanda_estimada']
})

# Concatena o DataFrame original com o novo DataFrame
unified_df = pd.concat([full_df, new_data], ignore_index=True)

# Remove as linhas com os SKUs especificados
skus_a_remover = [52594, 52595, 52600, 52601, 52605, 54526, 54528, 54532, 54601, 55750, 55751, 55752, 55753, 55754, 55755, 56090, 56435, 56436, 56437, 56438, 56492, 56493, 56494, 56495, 57033, 57377, 57392, 57393, 57396, 57397, 57398, 57399, 57400, 57401, 57402, 57403, 57404, 57405, 57406, 57407, 57408, 57409, 57410, 57413, 57424, 57904, 57905, 57906, 57907, 57908, 57916, 57917, 57918, 57919, 57920, 57921, 57922, 57924, 57925, 57926, 57928, 57929, 57930, 57931, 57932, 57933, 57934, 57935, 57936, 57937, 57938, 57939, 57940, 57941, 58360, 58361, 58362, 59325, 59935, 59936, 59937, 59938, 59939, 59940, 59946, 59947, 59948, 59949, 59950, 59951, 59952, 59953, 59955, 79378, 79379, 79380, 85443, 85444, 85445, 85446, 85447, 85549, 85550, 85551, 85552, 85553, 85643, 86390, 86391, 86392, 6393, 86771, 86773, 86775, 86777, 86781, 87839, 87840, 88131, 88132, 88133, 88338, 88339, 89086, 89087, 89088, 89089, 89152, 89330, 89502]
filtered_df = unified_df[~unified_df['sku_com_de_para'].isin(skus_a_remover)]
filtered_df = filtered_df.drop(index=0)


In [ ]:
nome_arquivo = f'tb_fill_rate_eud.csv'
nome_arquivo_saida = f'{nome_arquivo}'
sub_pasta_raiz = f'{sub_pasta}{nr_ciclo_pasta}{data_atual_string}/'
pasta_raiz_saida = f'{pasta_raiz}{sub_pasta_raiz}'
filtered_df.to_csv(f'{pasta_raiz_saida}{nome_arquivo_saida}', encoding='utf-8', index = False)

In [ ]:
import matplotlib.pyplot as plt
from tabulate import tabulate

# Filtrar o DataFrame para os canais desejados
df_canais = filtered_df[filtered_df['cod_canal'].isin(['VDF_AERO', 'VDF_FRANQ','LOJA_AERO','LOJA_GB'])].copy()

# Agrupar por cod_canal e somar as colunas relevantes
df_agregado = df_canais.groupby('cod_canal').agg(
    Pedido_Unit=('ponte_supply_demandado_unit', 'sum'),
    Atendido_Unit=('ponte_supply_vendas_unit', 'sum'),
    Pedido_Vlr=('ponte_supply_demandado_pvr', 'sum'),
    Atendido_Vlr=('ponte_supply_vendas_pvr', 'sum')
).reset_index()

# Calcular os totais gerais
total_geral = pd.DataFrame({
    'cod_canal': ['Total Geral'],
    'Pedido_Unit': [df_agregado['Pedido_Unit'].sum()],
    'Atendido_Unit': [df_agregado['Atendido_Unit'].sum()],
    'Pedido_Vlr': [df_agregado['Pedido_Vlr'].sum()],
    'Atendido_Vlr': [df_agregado['Atendido_Vlr'].sum()]
})

# Concatenar com os totais gerais
df_agregado = pd.concat([df_agregado, total_geral], ignore_index=True)

# Calcular as proporções, tratando divisão por zero
df_agregado['FR_Unit'] = df_agregado.apply(lambda row: f"{(row['Atendido_Unit'] / row['Pedido_Unit'] * 100):.1f}%" if row['Pedido_Unit'] != 0 else "0.0%", axis=1)
df_agregado['FR_Vlr'] = df_agregado.apply(lambda row: f"{(row['Atendido_Vlr'] / row['Pedido_Vlr'] * 100):.1f}%" if row['Pedido_Vlr'] != 0 else "0.0%", axis=1)

df_agregado['Pedido_Unit'] = df_agregado['Pedido_Unit'].apply(lambda n: f"{n/1e9:.1f}B" if abs(n) >= 1e9 else (
    f"{n/1e6:.1f}M" if abs(n) >= 1e6 else (
    f"{n/1e3:.1f}K" if abs(n) >= 1e3 else (
    f"{n}"
))))
df_agregado['Atendido_Unit'] = df_agregado['Atendido_Unit'].apply(lambda n: f"{n/1e9:.1f}B" if abs(n) >= 1e9 else (
    f"{n/1e6:.1f}M" if abs(n) >= 1e6 else (
    f"{n/1e3:.1f}K" if abs(n) >= 1e3 else (
    f"{n}"
))))
df_agregado['Pedido_Vlr'] = df_agregado['Pedido_Vlr'].apply(lambda n: f"{n/1e9:.1f}B" if abs(n) >= 1e9 else (
    f"{n/1e6:.1f}M" if abs(n) >= 1e6 else (
    f"{n/1e3:.1f}K" if abs(n) >= 1e3 else (
    f"{n}"
))))
df_agregado['Atendido_Vlr'] = df_agregado['Atendido_Vlr'].apply(lambda n: f"{n/1e9:.1f}B" if abs(n) >= 1e9 else (
    f"{n/1e6:.1f}M" if abs(n) >= 1e6 else (
    f"{n/1e3:.1f}K" if abs(n) >= 1e3 else (
    f"{n}"
))))
# Exibir o DataFrame resultante
print(tabulate(df_agregado, headers='keys', tablefmt='psql'))


+----+-------------+---------------+-----------------+--------------+----------------+-----------+----------+
|    | cod_canal   | Pedido_Unit   | Atendido_Unit   | Pedido_Vlr   | Atendido_Vlr   | FR_Unit   | FR_Vlr   |
|----+-------------+---------------+-----------------+--------------+----------------+-----------+----------|
|  0 | LOJA_AERO   | 50.5K         | 36.9K           | 982.3K       | 751.5K         | 73.0%     | 76.5%    |
|  1 | LOJA_GB     | 6.5K          | 3.6K            | 129.4K       | 77.1K          | 55.8%     | 59.6%    |
|  2 | VDF_AERO    | 769.6K        | 496.4K          | 16.2M        | 11.4M          | 64.5%     | 70.1%    |
|  3 | VDF_FRANQ   | 5.7M          | 3.7M            | 109.0M       | 72.8M          | 64.0%     | 66.7%    |
|  4 | Total Geral | 6.6M          | 4.2M            | 126.4M       | 85.0M          | 64.1%     | 67.2%    |
+----+-------------+---------------+-----------------+--------------+----------------+-----------+----------+
